# Webscraping Anticipa — Notebook consolidado

Reúne en un solo notebook los 7 procesos de extracción. Todas las salidas quedan en la carpeta de Google Drive que definas en `RUTA_OUTPUT` (celda de configuración):

```
<RUTA_OUTPUT>/
├── resumen_ejecucion.json              Resultado de la última ejecución
├── registro_archivos.csv               Fechas de cada archivo y de su borrado
├── 01_OSPA_IDEAM/
│   ├── datos_originales/               CSV y shapefiles descargados
│   ├── pdf_para_rag/                   Un PDF por fuente y día: <fuente>_AAAA-MM-DD.pdf
│   ├── huellas_ospa.json               Control de cambios por fuente y día (no borrar)
│   └── reporte_alertas_ideam.xlsx      Control de descargas
├── 02_Boletines_IDEAM/
│   ├── pdfs/                           Boletines en PDF
│   ├── manifest.csv                    Control de lo descargado (no borrar)
│   └── logs/                           Historial y resumen
├── 03_Monitor_Amazonia/
│   └── <País>/<País>_<Evento>_AAAA-MM-DD.pdf    Un PDF por evento
├── inamhi_elnino/
│   └── inamhi_elnino_ultimo_informe.pdf    Último informe ERFEN (se sobrescribe)
├── inamhi_boletines/
│   └── <Boletín>_AAAA-MM-DD.pdf             Cada versión publicada (no se borra)
├── inamhi_pronostico_diario/
│   ├── inamhi_pronostico_AAAA-MM-DD.json   Pronóstico por ciudad, para Anticipa
│   └── inamhi_pronostico_AAAA-MM-DD.pdf    El mismo pronóstico, por provincia y ciudad
└── senamhi_alertas_diaria/
    ├── senamhi_alertas_diaria.json         Avisos activos del SENAMHI (se sobrescribe)
    └── senamhi_aviso_<Nro>.pdf             Un PDF por aviso activo, solo texto
```

| Módulo | Qué hace | Fuente |
|---|---|---|
| 1. OSPA IDEAM | Descarga alertas y modelos, y genera PDF para el RAG | `bart.ideam.gov.co` |
| 2. Boletines IDEAM | Descarga solo los boletines nuevos | `www.ideam.gov.co` |
| 3. Monitor Amazonía | Genera un PDF por evento, agrupado por país (sin consultar el IDEAM) | GitHub `3is-global/echo_dp_amazon_monitor` |
| 4. INAMHI El Niño | Descarga el último informe ERFEN y sobrescribe el anterior | `servicios.inamhi.gob.ec/fenomeno-el-nino/` |
| 5. INAMHI Boletines | Descarga cada versión nueva de los boletines publicados | `servicios.inamhi.gob.ec/boletines/` |
| 6. INAMHI Pronóstico | Pronóstico por provincia y ciudad en JSON y PDF; solo queda el del día | `servicios.inamhi.gob.ec/pronostico/` |
| 7. SENAMHI Alertas | Avisos meteorológicos vigentes y emitidos de Perú, desde hoy: un JSON y un PDF por aviso | `www.senamhi.gob.pe/?p=aviso-meteorologico` |

**Uso:** Entorno de ejecución → Ejecutar todo. Los módulos son independientes: si uno falla, los demás siguen y el resumen final indica qué pasó. Para omitir un módulo, cámbialo a `False` en la celda de configuración.

## Retención de archivos

Un archivo se conserva si tiene `DIAS_RETENCION` días o menos (7 por defecto), o si es la publicación más reciente de su serie.

| Tipo | Qué pasa |
|---|---|
| Diario | Se borra pasados los 7 días |
| Semanal | Se borra cuando ya no es el último y tiene más de 7 días |
| Mensual, trimestral o anual | Se queda hasta que llegue el siguiente |
| Sin fecha en el nombre | Se queda solo el último de su repositorio |
| Monitor (PDF por evento) | Se queda solo la última versión de cada evento |
| OSPA (PDF por fuente y día) | Se borra pasados los 7 días, salvo el último de cada fuente |

Con `BORRAR_ARCHIVOS = False` no se borra nada: `registro_archivos.csv` marca como `por_borrar` lo que se eliminaría. Revísalo y cámbialo a `True` para activar el borrado. El borrado solo afecta a Drive.

Ningún PDF se reescribe, salvo el del módulo INAMHI El Niño, que siempre es el último informe, el pronóstico diario de INAMHI Pronóstico, que se actualiza durante el día y se borra al día siguiente, y las alertas de SENAMHI, que se sobrescriben con los avisos activos. Si cambian los datos de un día ya publicado, se crea otro PDF con la marca de tiempo en que se obtuvo: `<fuente>_AAAA-MM-DD_obtenido_AAAA-MM-DD_HHMMSS.pdf`. En Drive se queda la versión más reciente de cada día.

In [ ]:
# 1. Dependencias
%pip -q install requests beautifulsoup4 pandas openpyxl reportlab

In [ ]:
# 2. Configuración general
import csv
import hashlib
import json
import os
import tempfile
import time
import traceback
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

from IPython.display import display

# >>> DEFINE AQUÍ TU CARPETA DE DRIVE <<<
# Unidad compartida: "/content/drive/Shareddrives/<Nombre>/<Carpeta>"
RUTA_OUTPUT = "/content/drive/MyDrive/Anticipa Webscrapping/OUTPUT"

# Retención: se conserva lo que tenga estos días o menos,
# y siempre la publicación más reciente de cada serie.
DIAS_RETENCION = 7

# False = no borra nada; solo marca "por_borrar" en el registro.
# True  = borra de Drive los archivos vencidos.
BORRAR_ARCHIVOS = True

# Módulos que se ejecutan con "Ejecutar todo"
EJECUTAR_OSPA = True
EJECUTAR_BOLETINES = True
EJECUTAR_MONITOR = True
EJECUTAR_INAMHI = True
EJECUTAR_INAMHI_BOLETINES = True
EJECUTAR_INAMHI_PRONOSTICO = True
# SENAMHI rechaza las conexiones desde GitHub Actions (tiempo de espera agotado);
# desde un PC en la región sí responde. Apagado en GitHub hasta definir otra vía.
EJECUTAR_SENAMHI = False

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    if not RUTA_OUTPUT.startswith("/content/drive/"):
        raise ValueError("RUTA_OUTPUT debe empezar por /content/drive/ para guardarse en Drive")
    OUTPUT = Path(RUTA_OUTPUT)
except ImportError:
    # Fuera de Colab: carpeta local (o la indicada en OUTPUT_WEBSCRAPING)
    OUTPUT = Path(os.environ.get("OUTPUT_WEBSCRAPING", Path.cwd() / "OUTPUT"))

DIR_OSPA = OUTPUT / "01_OSPA_IDEAM"
DIR_BOLETINES = OUTPUT / "02_Boletines_IDEAM"
DIR_MONITOR = OUTPUT / "03_Monitor_Amazonia"
DIR_INAMHI = OUTPUT / "inamhi_elnino"
DIR_INAMHI_BOLETINES = OUTPUT / "inamhi_boletines"
DIR_INAMHI_PRONOSTICO = OUTPUT / "inamhi_pronostico_diario"
DIR_SENAMHI = OUTPUT / "senamhi_alertas_diaria"

for carpeta in (DIR_OSPA, DIR_BOLETINES, DIR_MONITOR, DIR_INAMHI, DIR_INAMHI_BOLETINES,
                DIR_INAMHI_PRONOSTICO, DIR_SENAMHI):
    carpeta.mkdir(parents=True, exist_ok=True)

# ----- Registro de fechas de todos los archivos -----
REGISTRO_PATH = OUTPUT / "registro_archivos.csv"
REGISTRO_COLUMNAS = [
    "modulo", "serie", "archivo", "fecha_archivo", "periodo", "primera_vez",
    "ultima_revision", "estado", "fecha_borrado", "motivo", "huella",
]
TZ_COLOMBIA = ZoneInfo("America/Bogota")

REGISTRO = {}
if REGISTRO_PATH.exists():
    with open(REGISTRO_PATH, encoding="utf-8-sig", newline="") as f:
        for fila in csv.DictReader(f):
            REGISTRO[(fila["modulo"], fila["archivo"])] = fila


def registrar(modulo, serie, archivo, fecha="", periodo="", estado="vigente", motivo="", huella=None):
    """Crea o actualiza la fila de un archivo. La historia no se borra."""
    ahora = datetime.now(TZ_COLOMBIA).isoformat(timespec="seconds")
    fila = REGISTRO.setdefault((modulo, archivo), {c: "" for c in REGISTRO_COLUMNAS})
    fila.update(modulo=modulo, serie=serie, archivo=archivo, ultima_revision=ahora,
                estado=estado, motivo=motivo)
    fila["fecha_archivo"] = fecha or fila["fecha_archivo"]
    fila["periodo"] = periodo or fila["periodo"]
    fila["primera_vez"] = fila["primera_vez"] or ahora
    if huella is not None:
        fila["huella"] = huella
    if estado == "borrado":
        fila["fecha_borrado"] = fila["fecha_borrado"] or ahora


def guardar_registro():
    temporal = REGISTRO_PATH.with_suffix(".csv.tmp")
    with open(temporal, "w", encoding="utf-8-sig", newline="") as f:
        escritor = csv.DictWriter(f, fieldnames=REGISTRO_COLUMNAS)
        escritor.writeheader()
        escritor.writerows(REGISTRO.values())
    os.replace(temporal, REGISTRO_PATH)


modo = "BORRA archivos vencidos" if BORRAR_ARCHIVOS else "SIMULACIÓN: no borra nada"
print("Las salidas se guardarán en:", OUTPUT)
print(f"Retención: {DIAS_RETENCION} días | {modo}")

## 3. Módulo 1: OSPA IDEAM

Descarga alertas y modelos de `bart.ideam.gov.co` y genera un PDF por fuente y día para el RAG, con una línea autoexplicativa por registro. Cada PDF lleva la fecha en el nombre y solo trae los datos de ese día, y nunca se reescribe: si esos datos cambian, se crea otro PDF con la marca de tiempo en que se obtuvo. En hidrología se usa la última corrida de cada día. Parámetros principales: `SOLO_HOY`, `MAX_TOTAL_MB`, `FUENTES` y `PAGINAS_POR_PDF`.

In [ ]:
# 3. Módulo 1: OSPA IDEAM
def ejecutar_ospa():
    from pathlib import Path
    from datetime import datetime
    from zoneinfo import ZoneInfo
    from email.utils import parsedate_to_datetime
    from urllib.parse import urljoin, urlsplit, unquote
    from bs4 import BeautifulSoup
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill
    from urllib.robotparser import RobotFileParser
    from IPython.display import display
    import pandas as pd
    import requests
    import hashlib
    import os
    import time

    # ==================================================
    # 1. CONFIGURACIÓN
    # ==================================================

    BASE = "https://bart.ideam.gov.co/ospa/"
    TZ = ZoneInfo("America/Bogota")

    SOLO_HOY = False   # True = solo archivos modificados hoy
    MAX_TOTAL_MB = 100
    MAX_ARCHIVO_MB = 30
    PAUSA = 1

    # Carpetas de alertas y modelos (incluye Meteomarinas)
    FUENTES = {
        "Deslizamientos": [
            "DatosAbiertos/Alertas_Deslizamientos/",
            "Alertas/modelos/deslizamientos/ultimo/"
        ],
        "Incendios": [
            "DatosAbiertos/Alertas_Incendios/",
            "Alertas/modelos/incendios/ultimo/"
        ],
        "Hidrología": [
            "DatosAbiertos/Alertas_Hidrologicas/"
        ],
        "Meteomarinas": [
            "DatosAbiertos/Alertas_Meteomarinas/"
        ]
    }

    EXTENSIONES = {
        ".csv", ".pdf", ".json", ".geojson",
        ".shp", ".shx", ".dbf", ".prj"
    }

    # ==================================================
    # 2. CARPETAS DE SALIDA
    # ==================================================

    DESTINO = DIR_OSPA / "datos_originales"
    DESTINO.mkdir(parents=True, exist_ok=True)

    REPORTE = DIR_OSPA / "reporte_alertas_ideam.xlsx"

    # ==================================================
    # 3. SESIÓN Y RESTRICCIONES
    # ==================================================

    session = requests.Session()
    USER_AGENT = "IDEAM-OSPA-Research/1.0"

    session.headers.update({"User-Agent": USER_AGENT})

    # Reintenta fallos de red o SSL pasajeros, sin desactivar la verificación
    session.mount("https://", requests.adapters.HTTPAdapter(
        max_retries=requests.adapters.Retry(
            total=3, status=0, backoff_factor=2
        )
    ))

    robots = RobotFileParser()
    robots_activos = False

    respuesta = session.get(
        "https://bart.ideam.gov.co/robots.txt",
        timeout=20
    )

    if respuesta.status_code == 200:
        robots.parse(respuesta.text.splitlines())
        robots_activos = True
    elif respuesta.status_code != 404:
        raise RuntimeError(
            f"No se pudo verificar robots.txt: "
            f"HTTP {respuesta.status_code}"
        )

    def permitido(url):
        return (
            not robots_activos
            or robots.can_fetch(USER_AGENT, url)
        )

    def fecha_servidor(valor):
        if not valor:
            return None
        try:
            return parsedate_to_datetime(valor).astimezone(TZ)
        except (ValueError, TypeError):
            return None

    def hash_archivo(ruta):
        h = hashlib.sha256()
        with open(ruta, "rb") as f:
            for bloque in iter(
                lambda: f.read(1024 * 1024), b""
            ):
                h.update(bloque)
        return h.hexdigest()

    # ==================================================
    # 4. DESCUBRIR ARCHIVOS EN LAS CARPETAS CONFIGURADAS
    # ==================================================

    archivos = []
    errores_exploracion = []
    hoy = datetime.now(TZ).date()
    detener = False

    for categoria, carpetas in FUENTES.items():

        for subcarpeta in carpetas:

            url_carpeta = urljoin(BASE, subcarpeta)

            print(f"\nExplorando: {categoria} / {subcarpeta}")

            if not permitido(url_carpeta):
                errores_exploracion.append({
                    "Carpeta": subcarpeta,
                    "Error": "Restringida por robots.txt"
                })
                continue

            try:
                r = session.get(
                    url_carpeta,
                    timeout=(10, 30)
                )

                if r.status_code in (403, 429):
                    detener = True
                    errores_exploracion.append({
                        "Carpeta": subcarpeta,
                        "Error": f"HTTP {r.status_code}"
                    })
                    break

                r.raise_for_status()
                r.encoding = "iso-8859-1"

                soup = BeautifulSoup(r.text, "html.parser")

                for a in soup.find_all("a", href=True):

                    href = a["href"]

                    if (
                        href.startswith("?")
                        or href.startswith("../")
                        or href.startswith("#")
                    ):
                        continue

                    url = urljoin(url_carpeta, href)
                    parsed = urlsplit(url)

                    # No salir de la carpeta seleccionada
                    if (
                        parsed.scheme != "https"
                        or parsed.netloc != "bart.ideam.gov.co"
                        or not parsed.path.startswith(
                            urlsplit(url_carpeta).path
                        )
                        or parsed.query
                        or parsed.fragment
                    ):
                        continue

                    # No navegar a otras subcarpetas
                    if parsed.path.endswith("/"):
                        continue

                    relativo = unquote(
                        parsed.path[len(urlsplit(BASE).path):]
                    )

                    partes = Path(relativo).parts

                    if any(
                        p in (".", "..")
                        or "\\" in p
                        or "\x00" in p
                        for p in partes
                    ):
                        continue

                    if Path(relativo).suffix.lower() not in EXTENSIONES:
                        continue

                    archivos.append({
                        "Categoria": categoria,
                        "Archivo": Path(relativo).name,
                        "Ruta relativa": relativo,
                        "URL": url
                    })

                print("Archivos identificados en esta carpeta.")

            except requests.RequestException as e:
                errores_exploracion.append({
                    "Carpeta": subcarpeta,
                    "Error": str(e)
                })

            time.sleep(PAUSA)

        if detener:
            break

    # Evitar URLs duplicadas
    archivos = list({
        a["URL"]: a for a in archivos
    }.values())

    print("\nTOTAL ARCHIVOS IDENTIFICADOS:", len(archivos))

    # ==================================================
    # 5. DESCARGAR Y VALIDAR
    # ==================================================

    resultados = []
    total_bytes = 0

    MAX_TOTAL = MAX_TOTAL_MB * 1024 * 1024
    MAX_ARCHIVO = MAX_ARCHIVO_MB * 1024 * 1024

    for info in archivos if not detener else []:

        url = info["URL"]
        ruta = DESTINO / info["Ruta relativa"]
        tmp = ruta.with_name(ruta.name + ".part")

        estado = "ERROR"
        http = None
        fecha_modificacion = None
        tamano = 0
        mensaje = ""
        checksum = ""

        try:
            if not permitido(url):
                estado = "RESTRINGIDO"
                mensaje = "Restricción robots.txt"

            else:
                head = session.head(
                    url,
                    timeout=(10, 30),
                    allow_redirects=True
                )

                http = head.status_code

                if http in (403, 429):
                    detener = True
                    estado = "BLOQUEADO"
                    mensaje = f"HTTP {http}"

                else:
                    head.raise_for_status()

                    fecha_modificacion = fecha_servidor(
                        head.headers.get("Last-Modified")
                    )

                    if SOLO_HOY and (
                        fecha_modificacion is None
                        or fecha_modificacion.date() != hoy
                    ):
                        estado = "OTRA FECHA"
                        mensaje = (
                            "No modificado hoy o fecha desconocida"
                        )

                    else:
                        longitud = head.headers.get(
                            "Content-Length"
                        )

                        if longitud and int(longitud) > MAX_ARCHIVO:
                            estado = "MUY GRANDE"
                            mensaje = "Supera límite por archivo"

                        elif longitud and (
                            total_bytes + int(longitud) > MAX_TOTAL
                        ):
                            estado = "LÍMITE"
                            mensaje = "Supera límite total"

                        else:
                            ruta.parent.mkdir(
                                parents=True,
                                exist_ok=True
                            )

                            with session.get(
                                url,
                                stream=True,
                                timeout=(10, 90)
                            ) as r:

                                http = r.status_code

                                if http in (403, 429):
                                    detener = True
                                    estado = "BLOQUEADO"
                                    raise RuntimeError(
                                        f"HTTP {http}"
                                    )

                                r.raise_for_status()

                                if "text/html" in r.headers.get(
                                    "Content-Type", ""
                                ).lower():
                                    raise ValueError(
                                        "Respuesta HTML inesperada"
                                    )

                                esperado = r.headers.get(
                                    "Content-Length"
                                )

                                h = hashlib.sha256()

                                with open(tmp, "wb") as f:
                                    for bloque in r.iter_content(
                                        262144
                                    ):
                                        if not bloque:
                                            continue

                                        tamano += len(bloque)

                                        if (
                                            tamano > MAX_ARCHIVO
                                            or total_bytes + tamano > MAX_TOTAL
                                        ):
                                            raise ValueError(
                                                "Límite de tamaño alcanzado"
                                            )

                                        f.write(bloque)
                                        h.update(bloque)

                                if tamano == 0:
                                    raise ValueError("Archivo vacío")

                                if esperado and tamano != int(esperado):
                                    raise ValueError(
                                        "Descarga incompleta"
                                    )

                                checksum = h.hexdigest()

                            total_bytes += tamano

                            if (
                                ruta.exists()
                                and hash_archivo(ruta) == checksum
                            ):
                                tmp.unlink()
                                estado = "SIN CAMBIOS"
                            else:
                                existia = ruta.exists()
                                os.replace(tmp, ruta)
                                estado = (
                                    "ACTUALIZADO" if existia
                                    else "CORRECTO"
                                )

                            mensaje = "Archivo verificado"

        except Exception as e:
            if estado != "BLOQUEADO":
                estado = "ERROR"
            mensaje = str(e)

        finally:
            if tmp.exists():
                tmp.unlink()

        resultados.append({
            "Categoría": info["Categoria"],
            "Archivo": info["Archivo"],
            "Ruta relativa": info["Ruta relativa"],
            "Estado": estado,
            "HTTP": http,
            "Fecha modificación": (
                fecha_modificacion.isoformat()
                if fecha_modificacion else ""
            ),
            "Fecha extracción": datetime.now(TZ).isoformat(),
            "Tamaño bytes": tamano,
            "SHA256": checksum,
            "URL origen": url,
            "Ruta Drive": str(ruta) if ruta.exists() else "",
            "Observaciones": mensaje
        })

        print(f"{info['Categoria']}: {info['Archivo']} -> {estado}")

        if detener:
            break

        time.sleep(PAUSA)

    session.close()

    # ==================================================
    # 6. CREAR REPORTE EXCEL
    # ==================================================

    wb = Workbook()
    ws = wb.active
    ws.title = "Control descargas"

    df = pd.DataFrame(resultados)

    if df.empty:
        df = pd.DataFrame(columns=[
            "Categoría", "Archivo", "Ruta relativa",
            "Estado", "HTTP", "Fecha modificación",
            "Fecha extracción", "Tamaño bytes",
            "SHA256", "URL origen", "Ruta Drive",
            "Observaciones"
        ])

    ws.append(df.columns.tolist())

    for fila in df.itertuples(index=False, name=None):
        ws.append([
            "" if pd.isna(v) else v
            for v in fila
        ])

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for c in ws[1]:
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor="17365D")

    for col in ws.columns:
        letra = col[0].column_letter
        ws.column_dimensions[letra].width = (
            65 if letra in ("J", "K") else 27
        )

    estados = {
        "CORRECTO": "D9EAD3",
        "ACTUALIZADO": "D9EAD3",
        "SIN CAMBIOS": "DDEBF7",
        "ERROR": "F4CCCC",
        "BLOQUEADO": "F4CCCC",
        "OTRA FECHA": "EEEEEE"
    }

    for fila in range(2, ws.max_row + 1):
        celda = ws.cell(fila, 4)
        if celda.value in estados:
            celda.fill = PatternFill(
                "solid",
                fgColor=estados[celda.value]
            )

    ws2 = wb.create_sheet("Errores exploracion")
    ws2.append(["Carpeta", "Error"])

    for e in errores_exploracion:
        ws2.append([e["Carpeta"], e["Error"]])

    temporal_excel = REPORTE.with_name(
        "reporte_alertas_ideam.tmp.xlsx"
    )

    wb.save(temporal_excel)
    os.replace(temporal_excel, REPORTE)

    # ==================================================
    # 7. RESUMEN
    # ==================================================

    print("\n========== RESULTADO ==========")

    print("Archivos identificados:", len(archivos))
    print("MB transferidos:", round(total_bytes / 1048576, 2))
    print("Reporte:", REPORTE)

    if not df.empty:
        display(
            df.groupby(
                ["Categoría", "Estado"]
            ).size().reset_index(name="Cantidad")
        )
        display(df[[
            "Categoría",
            "Archivo",
            "Estado",
            "Fecha modificación",
            "Tamaño bytes"
        ]])

    if detener:
        print(
            "ATENCIÓN: proceso interrumpido por "
            "restricción HTTP 403/429."
        )

    print("\nProceso terminado.")

    # ==================================================
    # 8. CONVERTIR A PDF PARA EL RAG
    # ==================================================
    # Cada registro se escribe como una línea que se explica
    # sola ("COLUMNA: valor | ..."), porque al trocear un PDF
    # con tablas el RAG pierde los encabezados de las columnas.

    import re
    import textwrap
    from datetime import date, timedelta
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfbase.pdfmetrics import stringWidth
    from reportlab.pdfgen.canvas import Canvas

    CARPETA_RAG = DIR_OSPA / "pdf_para_rag"
    CARPETA_RAG.mkdir(exist_ok=True)

    # Un archivo grande se parte en varios PDF de este tamaño.
    # Ajustar al límite del RAG.
    PAGINAS_POR_PDF = 500

    # Días de datos que se convierten en PDF. None = todo el histórico, sin borrar.
    # Los CSV completos siguen en datos_originales/.
    DIAS_RAG_OSPA = DIAS_RETENCION

    # Columnas-fecha de las matrices (hidrología, meteomarinas):
    # "1/01/2026", "1/01/2026 0:00" o "2026-11-04-16:13:15"
    PATRON_FECHA = re.compile(
        r"^(\d{1,2}/\d{1,2}/\d{4}|\d{4}-\d{2}-\d{2}-)"
    )

    FUENTE = "Helvetica"
    NEGRITA = "Helvetica-Bold"
    TAMANO = 7.5
    INTERLINEA = 10
    MARGEN = 36
    ANCHO_PAGINA, ALTO_PAGINA = A4
    ANCHO_UTIL = ANCHO_PAGINA - 2 * MARGEN
    CARACTERES_POR_LINEA = 95


    def leer_csv(ruta):
        # El IDEAM publica unos CSV en UTF-8 y otros en Latin-1
        for codificacion in ("utf-8-sig", "cp1252"):
            try:
                return pd.read_csv(
                    ruta, sep=";", dtype=str,
                    keep_default_na=False,
                    encoding=codificacion
                )
            except UnicodeDecodeError:
                continue
        raise ValueError("Codificación no reconocida")


    def ultima_corrida_por_dia(fechas):
        """De las columnas con hora (corridas del modelo) deja la
        última de cada día. Las columnas sin hora se conservan."""
        por_dia = {}
        for c in fechas:
            if ":" in c:
                a, b, d = re.findall(r"\d+", c)[:3]
                clave = (a, b, d) if len(a) == 4 else (d, b.zfill(2), a.zfill(2))
            else:
                clave = c
            por_dia[clave] = c
        return list(por_dia.values())


    def interpretar_fechas(valores):
        """Convierte las fechas de una columna. El IDEAM usa dd/mm/aaaa o
        mm/dd/aaaa según el archivo: si un valor lo aclara (un número mayor
        que 12), vale para toda la columna; si no, se usa la opción más
        cercana a hoy que no esté en el futuro."""
        partes = {
            v: [int(x) for x in v.split()[0].split("/")]
            for v in valores if v.strip()
        }
        dia_primero = any(p[0] > 12 for p in partes.values())
        mes_primero = any(p[1] > 12 for p in partes.values())

        def leer_fecha(p):
            opciones = []
            if not mes_primero and p[1] <= 12:
                opciones.append(date(p[2], p[1], p[0]))
            if not dia_primero and p[0] <= 12:
                opciones.append(date(p[2], p[0], p[1]))
            posibles = [o for o in opciones if o <= hoy] or opciones
            return min(posibles, key=lambda o: abs((hoy - o).days))

        return {v: leer_fecha(p) for v, p in partes.items()}


    def publicaciones(datos, fecha_por_defecto):
        """Separa los registros por fecha y devuelve [(fecha, líneas)].

        Cada línea se explica sola. En las matrices hay una línea por cada
        valor no vacío. El día en curso de las columnas con hora se omite
        porque aún pueden llegar corridas de ese día."""
        datos = datos.drop(columns=[
            c for c in datos.columns
            if c.startswith("Unnamed") and (datos[c] == "").all()
        ])
        todas = [c for c in datos.columns if PATRON_FECHA.match(c)]
        fijas = [c for c in datos.columns if c not in todas]
        columna_fecha = next(
            (c for c in ("FECHA_EJECUCION", "FECHA") if c in fijas), None
        )

        grupos = {}
        if todas:
            for c in ultima_corrida_por_dia(todas):
                if re.match(r"\d{4}-", c):
                    fecha = date.fromisoformat(c[:10])
                else:
                    fecha = datetime.strptime(
                        c.split()[0].split(".")[0], "%d/%m/%Y"
                    ).date()
                if ":" in c and fecha == hoy:
                    continue
                grupos.setdefault(fecha, []).append(c)
        elif columna_fecha:
            mapa = interpretar_fechas(datos[columna_fecha].unique())
            for valor, fecha in mapa.items():
                grupos.setdefault(fecha, []).append(valor)
        else:
            grupos[fecha_por_defecto] = []

        if not grupos:
            return []

        def base(columnas, fila):
            return " | ".join(
                f"{c}: {str(v).strip()}"
                for c, v in zip(columnas, fila)
                if str(v).strip()
            )

        salida = []
        for fecha in sorted(grupos):
            lineas = []
            if todas:
                columnas = grupos[fecha]
                # pandas renombra las fechas repetidas como "6/08/2026.1"
                etiquetas = [re.sub(r"\.\d+$", "", c) for c in columnas]
                for fila in datos[fijas + columnas].itertuples(index=False, name=None):
                    inicio = base(fijas, fila)
                    for etiqueta, valor in zip(etiquetas, fila[len(fijas):]):
                        if str(valor).strip():
                            lineas.append(
                                f"{inicio} | FECHA: {etiqueta} | VALOR: {str(valor).strip()}"
                            )
            else:
                parte = (
                    datos[datos[columna_fecha].isin(grupos[fecha])]
                    if columna_fecha else datos
                )
                lineas = [
                    base(parte.columns, fila)
                    for fila in parte.itertuples(index=False, name=None)
                ]
            salida.append((fecha, lineas))
        return salida


    def escribir_pdfs(lineas, carpeta, nombre, titulo):
        """Escribe las líneas en uno o más PDF nuevos.
        Devuelve registros y PDF generados."""
        temporales = []
        lienzo = None
        paginas = 0
        y = 0
        registros = 0

        def abrir_parte():
            ruta = carpeta / f"{nombre}.{len(temporales) + 1:03d}.tmp"
            temporales.append(ruta)
            return Canvas(str(ruta), pagesize=A4)

        def iniciar_pagina():
            lienzo.setFont(NEGRITA, 8)
            lienzo.drawString(MARGEN, ALTO_PAGINA - MARGEN, titulo)
            lienzo.setFont(FUENTE, TAMANO)
            return ALTO_PAGINA - MARGEN - 2 * INTERLINEA

        try:
            for linea in lineas:
                registros += 1
                trozos = (
                    [linea]
                    if stringWidth(linea, FUENTE, TAMANO) <= ANCHO_UTIL
                    else textwrap.wrap(linea, CARACTERES_POR_LINEA)
                )
                # Un registro no se corta entre dos páginas
                if lienzo is None or y - INTERLINEA * len(trozos) < MARGEN:
                    if lienzo is not None:
                        lienzo.showPage()
                        paginas += 1
                        if paginas >= PAGINAS_POR_PDF:
                            lienzo.save()
                            lienzo = None
                            paginas = 0
                    if lienzo is None:
                        lienzo = abrir_parte()
                    y = iniciar_pagina()

                for trozo in trozos:
                    lienzo.drawString(MARGEN, y, trozo)
                    y -= INTERLINEA

            if lienzo is None:
                raise ValueError("Sin registros")

            lienzo.showPage()
            lienzo.save()

            for i, tmp in enumerate(temporales, 1):
                final = (
                    f"{nombre}.pdf" if len(temporales) == 1
                    else f"{nombre}_parte_{i:03d}.pdf"
                )
                os.replace(tmp, carpeta / final)

        finally:
            for tmp in temporales:
                if tmp.exists():
                    tmp.unlink()

        return registros, len(temporales)


    def pdfs_de(nombre):
        return [
            *CARPETA_RAG.glob(f"{nombre}.pdf"),
            *CARPETA_RAG.glob(f"{nombre}_parte_*.pdf"),
        ]


    HUELLAS_PATH = DIR_OSPA / "huellas_ospa.json"
    huellas = (
        json.loads(HUELLAS_PATH.read_text(encoding="utf-8"))
        if HUELLAS_PATH.exists() else {}
    )
    marca_obtencion = datetime.now(TZ).strftime("%Y-%m-%d_%H%M%S")
    corte_ospa = (
        None if DIAS_RAG_OSPA is None
        else hoy - timedelta(days=DIAS_RAG_OSPA)
    )


    def guardar_huellas():
        temporal = HUELLAS_PATH.with_suffix(".json.tmp")
        temporal.write_text(json.dumps(huellas, indent=0), encoding="utf-8")
        os.replace(temporal, HUELLAS_PATH)


    pdf_resumen = []

    for fila in resultados:

        ruta_csv = DESTINO / fila["Ruta relativa"]

        if (
            ruta_csv.suffix.lower() != ".csv"
            or fila["Estado"] not in (
                "CORRECTO", "ACTUALIZADO", "SIN CAMBIOS"
            )
        ):
            continue

        fuente = ruta_csv.stem
        conocidas = huellas.setdefault(fuente, {})
        nuevos = 0
        cambios = 0
        registros = 0
        estado_pdf = "OK"

        try:
            fecha_defecto = (
                date.fromisoformat(fila["Fecha modificación"][:10])
                if fila["Fecha modificación"] else hoy
            )
            con_datos = [
                (f, l) for f, l in publicaciones(leer_csv(ruta_csv), fecha_defecto) if l
            ]
            ultima_fecha = max((f for f, _ in con_datos), default=None)

            for fecha_pub, lineas in con_datos:
                clave = fecha_pub.isoformat()
                huella = hashlib.sha256(
                    "\n".join(sorted(lineas)).encode("utf-8")
                ).hexdigest()
                anterior = conocidas.get(clave)
                nombre_base = f"{fuente}_{clave}"

                if anterior == huella:
                    continue

                if anterior is not None:
                    # Cambiaron los datos de un día ya visto: otro PDF con la
                    # marca de tiempo. El anterior no se reescribe.
                    nombre_pdf = f"{nombre_base}_obtenido_{marca_obtencion}"
                    cambios += 1
                elif pdfs_de(nombre_base):
                    conocidas[clave] = huella
                    continue
                elif (
                    corte_ospa is None
                    or fecha_pub >= corte_ospa
                    or fecha_pub == ultima_fecha
                ):
                    nombre_pdf = nombre_base
                    nuevos += 1
                else:
                    # Histórico antiguo: se anota para detectar futuras
                    # correcciones, sin generar PDF.
                    conocidas[clave] = huella
                    continue

                total, _ = escribir_pdfs(
                    lineas,
                    CARPETA_RAG,
                    nombre_pdf,
                    f"IDEAM OSPA - {fila['Categoría']} - {nombre_pdf}"
                )
                registros += total
                conocidas[clave] = huella

            # Retención: la versión más reciente de cada día se conserva 7 días
            # desde su fecha u obtención, y siempre el último día de la fuente.
            patron_pdf = re.compile(
                rf"^{re.escape(fuente)}_(\d{{4}}-\d{{2}}-\d{{2}})"
                r"(?:_obtenido_(\d{4}-\d{2}-\d{2}_\d{6}))?"
                r"(?:_parte_\d+)?\.pdf$"
            )
            versiones = {}
            for pdf in CARPETA_RAG.glob(f"{fuente}_*.pdf"):
                coincide = patron_pdf.match(pdf.name)
                if coincide:
                    versiones.setdefault(coincide.group(1), {}).setdefault(
                        coincide.group(2) or "", []
                    ).append(pdf)

            ultima_pdf = max(versiones, default=None)

            for fecha_pdf, por_marca in versiones.items():
                vigente = max(por_marca)
                for marca, lista in por_marca.items():
                    referencia = marca[:10] or fecha_pdf
                    if marca != vigente:
                        motivo = "Reemplazado por una versión con datos más recientes"
                    elif (
                        corte_ospa is None
                        or referencia >= corte_ospa.isoformat()
                        or fecha_pdf == ultima_pdf
                    ):
                        motivo = ""
                    else:
                        motivo = f"Más de {DIAS_RAG_OSPA} días y no es el último de su fuente"

                    for pdf in lista:
                        datos_registro = dict(
                            fecha=fecha_pdf, periodo="diario por fuente"
                        )
                        if not motivo:
                            registrar("OSPA", fila["Categoría"], pdf.name, **datos_registro)
                        elif BORRAR_ARCHIVOS:
                            pdf.unlink(missing_ok=True)
                            registrar("OSPA", fila["Categoría"], pdf.name,
                                      estado="borrado", motivo=motivo, **datos_registro)
                        else:
                            registrar("OSPA", fila["Categoría"], pdf.name,
                                      estado="por_borrar", motivo=motivo, **datos_registro)

        except Exception as e:
            estado_pdf = f"ERROR: {e}"

        guardar_huellas()

        pdf_resumen.append({
            "Archivo": ruta_csv.name,
            "Registros nuevos": registros,
            "PDF de días nuevos": nuevos,
            "PDF por cambios": cambios,
            "Estado": estado_pdf
        })
        print(f"{ruta_csv.name}: {nuevos} PDF de días nuevos, "
              f"{cambios} por cambios -> {estado_pdf}")

    print("\nCarpeta para subir al RAG:", CARPETA_RAG)
    display(pd.DataFrame(pdf_resumen))


    # ==================================================
    # 9. RESULTADO DEL MÓDULO
    # ==================================================

    descargas_con_error = sum(
        r["Estado"] in ("ERROR", "BLOQUEADO", "RESTRINGIDO", "LÍMITE", "MUY GRANDE")
        for r in resultados
    )
    pdf_con_error = sum(r["Estado"].startswith("ERROR") for r in pdf_resumen)

    return {
        "archivos_identificados": len(archivos),
        "pdf_para_rag": len(list(CARPETA_RAG.glob("*.pdf"))),
        "avisos": (
            len(errores_exploracion) + descargas_con_error
            + pdf_con_error + int(detener)
        ),
    }


## 4. Módulo 2: Boletines IDEAM

Descarga los boletines nuevos de `www.ideam.gov.co`. `manifest.csv` registra lo ya descargado para no repetirlo; no lo borres. Parámetros principales: `LOOKBACK_DAYS` y `DRY_RUN`.

In [ ]:
# 4. Módulo 2: Boletines IDEAM
def ejecutar_boletines():
    from datetime import datetime, timedelta
    from pathlib import Path
    from zoneinfo import ZoneInfo

    OUTPUT_DIR = DIR_BOLETINES
    PDF_DIR = OUTPUT_DIR / "pdfs"
    LOG_DIR = OUTPUT_DIR / "logs"
    TEMP_DIR = Path(tempfile.gettempdir()) / "ideam_download_tmp"
    MANIFEST_PATH = OUTPUT_DIR / "manifest.csv"
    RUN_LOG_PATH = LOG_DIR / "downloads.log"
    SUMMARY_PATH = LOG_DIR / "run_summary.json"
    CANDIDATES_PATH = LOG_DIR / "latest_candidates.csv"
    REVIEW_PATH = LOG_DIR / "undated_candidates.csv"

    for folder in (OUTPUT_DIR, PDF_DIR, LOG_DIR, TEMP_DIR):
        folder.mkdir(parents=True, exist_ok=True)

    TIMEZONE = ZoneInfo("America/Bogota")
    TODAY = datetime.now(TIMEZONE).date()
    LOOKBACK_DAYS = 2
    REQUEST_DELAY_SECONDS = 1.2
    MAX_PAGES_PER_REPOSITORY = 1  # Deliberately no historical pagination
    DRY_RUN = False  # Set True to preview candidates without downloading
    print("Colombia date:", TODAY)
    print("Persistent output:", OUTPUT_DIR)


    import logging
    import random
    import time
    import requests
    from requests.exceptions import SSLError, ConnectionError, Timeout
    from urllib.robotparser import RobotFileParser
    from urllib.parse import urljoin, urlparse, unquote

    BASE_URL = "https://www.ideam.gov.co"
    LANDING_URL = BASE_URL + "/sala-de-prensa/boletines"
    USER_AGENT = "IDEAMBulletinCollector/1.0 (research archive; respectful crawler)"
    MAX_CONNECTION_ATTEMPTS = 4

    logger = logging.getLogger("ideam_collector")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    logger.handlers.clear()
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    file_handler = logging.FileHandler(RUN_LOG_PATH, encoding="utf-8")
    file_handler.setFormatter(formatter)
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

    session = requests.Session()
    session.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/pdf,*/*",
        "Connection": "close",
    })

    def request_with_retries(url, *, stream=False, timeout=45):
        """Retry transient TLS/network failures only, never bypass certificate checks."""
        nonlocal session
        for attempt in range(1, MAX_CONNECTION_ATTEMPTS + 1):
            try:
                return session.get(
                    url, timeout=(15, timeout), stream=stream,
                    allow_redirects=True, verify=True
                )
            except (SSLError, ConnectionError, Timeout) as exc:
                logger.warning("Connection failure attempt %s/%s: %s | %s",
                               attempt, MAX_CONNECTION_ATTEMPTS, url, exc)
                if attempt >= MAX_CONNECTION_ATTEMPTS:
                    raise RuntimeError(
                        "IDEAM HTTPS connection failed after limited retries. "
                        "Do not disable SSL verification. Retry later or investigate "
                        "Colab's network access to the IDEAM host."
                    ) from exc
                session.close()
                session = requests.Session()
                session.headers.update({
                    "User-Agent": USER_AGENT,
                    "Accept": "text/html,application/pdf,*/*",
                    "Connection": "close",
                })
                time.sleep(min(12, 2 ** attempt) + random.uniform(0, 0.5))

    # Read robots.txt first. If unavailable, stop safely instead of assuming permission.
    robots_url = BASE_URL + "/robots.txt"
    robots_response = request_with_retries(robots_url, timeout=30)
    try:
        robots_response.raise_for_status()
        if "text/plain" not in robots_response.headers.get("Content-Type", "").lower():
            raise RuntimeError("Unexpected robots.txt content type; stopping safely")
        robots = RobotFileParser()
        robots.parse(robots_response.text.splitlines())
    finally:
        robots_response.close()

    last_request_at = 0.0

    def safe_get(url, *, stream=False, timeout=45):
        nonlocal last_request_at
        if urlparse(url).netloc.lower() != "www.ideam.gov.co":
            raise ValueError("External host not allowed: " + url)
        if not robots.can_fetch(USER_AGENT, url):
            raise PermissionError("robots.txt disallows: " + url)
        elapsed = time.monotonic() - last_request_at
        if elapsed < REQUEST_DELAY_SECONDS:
            time.sleep(REQUEST_DELAY_SECONDS - elapsed)
        # Reserve pacing before attempting the connection, including failures.
        last_request_at = time.monotonic()
        response = request_with_retries(url, stream=stream, timeout=timeout)
        if urlparse(response.url).netloc.lower() != "www.ideam.gov.co":
            response.close()
            raise ValueError("Unexpected redirect host: " + response.url)
        if not robots.can_fetch(USER_AGENT, response.url):
            response.close()
            raise PermissionError("Redirect disallowed by robots.txt: " + response.url)
        if response.status_code in (401, 403, 429):
            status = response.status_code
            response.close()
            raise PermissionError(f"Access restricted (HTTP {status}): {url}")
        try:
            response.raise_for_status()
        except Exception:
            response.close()
            raise
        return response

    logger.info("Collector initialized; destination=%s", OUTPUT_DIR)


    import re
    import unicodedata
    from bs4 import BeautifulSoup

    def category_from_url(url):
        slug = unquote(urlparse(url).path.rstrip("/").split("/")[-1])
        slug = unicodedata.normalize("NFC", slug)
        return re.sub(r"[-_]+", " ", slug).strip() or "Unnamed category"

    landing = safe_get(LANDING_URL)
    landing_soup = BeautifulSoup(landing.text, "html.parser")
    repositories = {}
    for a in landing_soup.select("a[href]"):
        label = a.get_text(" ", strip=True).casefold()
        href = urljoin(LANDING_URL, a.get("href", ""))
        parsed = urlparse(href)
        if (parsed.netloc.lower() == "www.ideam.gov.co"
                and parsed.path.startswith("/sala-de-prensa/boletines/")
                and ("consultar repositorio" in label or "repositorio" in label)):
            repositories[href] = {"category": category_from_url(href), "url": href}

    repositories = list(repositories.values())
    logger.info("Discovered %s repositories", len(repositories))
    if not repositories:
        raise RuntimeError("No repositories found; inspect the landing page HTML before proceeding.")
    for repo in repositories:
        print("-", repo["category"])

    import re
    import calendar
    import unicodedata
    from datetime import date

    MONTHS = {"enero":1,"ene":1,"febrero":2,"feb":2,"marzo":3,"mar":3,
              "abril":4,"abr":4,"mayo":5,"may":5,"junio":6,"jun":6,
              "julio":7,"jul":7,"agosto":8,"ago":8,"septiembre":9,"setiembre":9,
              "sep":9,"sept":9,"set":9,"octubre":10,"oct":10,"noviembre":11,
              "nov":11,"diciembre":12,"dic":12,"dec":12}
    MONTH_PATTERN = "|".join(sorted(MONTHS, key=len, reverse=True))

    def normalize_name(text):
        text = unicodedata.normalize("NFKD", text.lower())
        text = "".join(ch for ch in text if not unicodedata.combining(ch))
        return re.sub(r"[^a-z0-9]+", "_", text).strip("_")

    def parse_filename_date(name):
        s = normalize_name(name)
        patterns = [
            (rf"(?:^|_)({MONTH_PATTERN})_(\d{{1,2}})_(?:de_)?(20\d{{2}})(?:_|$)", "mdy"),
            (rf"(?:^|_)(\d{{1,2}})_(?:de_)?({MONTH_PATTERN})_(?:de_)?(20\d{{2}})(?:_|$)", "dmy"),
            (r"(?:^|_)(20\d{2})_(\d{1,2})_(\d{1,2})(?:_|$)", "ymd"),
            (r"(?:^|_)(\d{1,2})_(\d{1,2})_(20\d{2})(?:_|$)", "dmy_numeric"),
            (r"(?:^|_)(\d{2})(\d{2})(20\d{2})(?:_|$)", "ddmmyyyy"),
        ]
        for regex, kind in patterns:
            m = re.search(regex, s)
            if not m:
                continue
            try:
                a,b,c = m.groups()
                if kind == "mdy": return date(int(c), MONTHS[a], int(b)), "day"
                if kind == "dmy": return date(int(c), MONTHS[b], int(a)), "day"
                if kind == "ymd": return date(int(a), int(b), int(c)), "day"
                return date(int(c), int(b), int(a)), "day"
            except (ValueError, KeyError):
                continue
        # Month-only names such as ..._octubre_2026.pdf
        m = re.search(rf"(?:^|_)({MONTH_PATTERN})_(20\d{{2}})(?:_|$)", s)
        if m:
            return date(int(m.group(2)), MONTHS[m.group(1)], 1), "month"
        m = re.search(rf"(?:^|_)(20\d{{2}})_({MONTH_PATTERN})(?:_|$)", s)
        if m:
            return date(int(m.group(1)), MONTHS[m.group(2)], 1), "month"
        return None, "unknown"



    import pandas as pd

    found = []
    discovery_errors = []
    for number, repo in enumerate(repositories, 1):
        try:
            response = safe_get(repo["url"])
            soup = BeautifulSoup(response.text, "html.parser")
            records = {}
            for a in soup.select("a[href]"):
                url = urljoin(response.url, a["href"])
                path = urlparse(url).path.lower()
                name = a.get_text(" ", strip=True)
                if ("/file-download/download/public/" not in path and not path.endswith(".pdf")):
                    continue
                # The endpoint can serve other document types, hence require a PDF filename.
                if not name.lower().endswith(".pdf"):
                    continue
                detected_date, granularity = parse_filename_date(name)
                records[url] = {"category": repo["category"], "repository_url": repo["url"],
                                "filename": name, "file_url": url,
                                "file_date": detected_date.isoformat() if detected_date else "",
                                "date_precision": granularity}
            found.extend(records.values())
            logger.info("Repository %s/%s: %s PDF links - %s", number, len(repositories),
                        len(records), repo["category"])
        except Exception as exc:
            discovery_errors.append({"category": repo["category"], "error": str(exc)})
            logger.error("Repository failed: %s | %s", repo["category"], exc)

    candidates = pd.DataFrame(found, columns=["category","repository_url","filename","file_url","file_date","date_precision"])
    candidates = candidates.drop_duplicates(subset=["file_url"])
    candidates.to_csv(CANDIDATES_PATH, index=False, encoding="utf-8-sig")
    candidates[candidates["date_precision"] == "unknown"].to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig")
    print(f"Repositories: {len(repositories)} | PDF links: {len(candidates)} | Errors: {len(discovery_errors)}")
    display(candidates.head(15))

    MANIFEST_COLUMNS = ["file_url","category","filename","file_date","date_precision",
                        "saved_path","sha256","downloaded_at","status","error","deleted_at"]
    if MANIFEST_PATH.exists():
        manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")
        for col in MANIFEST_COLUMNS:
            if col not in manifest.columns: manifest[col] = ""
        manifest = manifest[MANIFEST_COLUMNS]
    else:
        manifest = pd.DataFrame(columns=MANIFEST_COLUMNS)

    # "deleted" cuenta como completado: un archivo borrado no se vuelve a descargar.
    successful_urls = set(manifest.loc[manifest["status"].isin(["downloaded", "duplicate_content", "deleted"]), "file_url"])
    known_hashes = set(manifest.loc[manifest["status"].isin(["downloaded","duplicate_content"]), "sha256"]) - {""}
    initial_run = not any(manifest["status"].isin(["downloaded", "duplicate_content", "deleted"]))
    cutoff = TODAY - timedelta(days=LOOKBACK_DAYS)

    selection = []
    for category, group in candidates.groupby("category", sort=False):
        dated = group[group["file_date"] != ""].copy()
        current_latest = []
        if not dated.empty:
            # Month-granularity dates are comparable within a monthly series.
            top_date = max(dated["file_date"])
            current_latest = dated.loc[dated["file_date"] == top_date, "file_url"].tolist()
        else:
            current_latest = group.head(1)["file_url"].tolist()

        for row in group.to_dict("records"):
            if row["file_url"] in successful_urls:
                continue
            if initial_run:
                choose = row["file_url"] in current_latest
            else:
                date_is_recent = bool(row["file_date"]) and row["date_precision"] == "day" and row["file_date"] >= cutoff.isoformat()
                choose = date_is_recent or row["file_url"] in current_latest
            if choose:
                selection.append(row)

    selected = pd.DataFrame(selection, columns=candidates.columns)
    logger.info("Initial run=%s; candidates selected=%s; already completed=%s",
                initial_run, len(selected), len(successful_urls))
    print("First run:", initial_run, "| PDFs to download:", len(selected))
    display(selected[["category", "filename", "file_date"]].head(30))

    import hashlib
    import json
    import os
    import shutil
    from datetime import datetime
    from pathlib import Path
    from urllib.parse import urlparse


    def safe_filename(name):
        safe = re.sub(r"[^\w.()-]+", "_", name, flags=re.UNICODE).strip("._")
        return safe[:170] if safe else "unnamed.pdf"


    def write_manifest():
        temporary = MANIFEST_PATH.with_suffix(".csv.tmp")
        manifest.to_csv(temporary, index=False, encoding="utf-8-sig")
        os.replace(temporary, MANIFEST_PATH)


    def record_result(row, status, saved_path="", digest="", error=""):
        nonlocal manifest
        entry = {"file_url": row["file_url"], "category": row["category"],
                 "filename": row["filename"], "file_date": row["file_date"],
                 "date_precision": row["date_precision"], "saved_path": saved_path,
                 "sha256": digest, "downloaded_at": datetime.now(TIMEZONE).isoformat(),
                 "status": status, "error": error}
        manifest = pd.concat([manifest[manifest["file_url"] != row["file_url"]],
                              pd.DataFrame([entry])], ignore_index=True)
        write_manifest()

    stats = {"downloaded": 0, "duplicate_content": 0, "failed": 0, "preview": 0}
    for idx, row in enumerate(selected.to_dict("records"), 1):
        if DRY_RUN:
            logger.info("DRY RUN %s/%s: %s", idx, len(selected), row["filename"])
            stats["preview"] += 1
            continue

        # Add IDEAM's public file ID to prevent filename collisions.
        doc_id = urlparse(row["file_url"]).path.rstrip("/").split("/")[-1]
        filename = safe_filename(Path(row["filename"]).name)
        destination = PDF_DIR / (doc_id + "_" + filename)
        temporary = TEMP_DIR / (doc_id + ".part")
        try:
            digest = hashlib.sha256()
            with safe_get(row["file_url"], stream=True, timeout=120) as response:
                with open(temporary, "wb") as output:
                    first_chunk = True
                    total_size = 0
                    for chunk in response.iter_content(chunk_size=128 * 1024):
                        if not chunk:
                            continue
                        if first_chunk:
                            if not chunk.startswith(b"%PDF-"):
                                raise ValueError("The response does not contain a PDF signature")
                            first_chunk = False
                        digest.update(chunk)
                        output.write(chunk)
                        total_size += len(chunk)
                    if total_size == 0:
                        raise ValueError("Empty PDF response")
            sha = digest.hexdigest()
            if sha in known_hashes:
                record_result(row, "duplicate_content", digest=sha)
                stats["duplicate_content"] += 1
                logger.info("Duplicate content skipped: %s", filename)
            else:
                # Colab /content and mounted Google Drive are different filesystems.
                # Copy into Drive first; rename only WITHIN Drive after verification.
                drive_partial = destination.with_name(destination.name + ".part")
                try:
                    shutil.copyfile(temporary, drive_partial)
                    check = hashlib.sha256()
                    with open(drive_partial, "rb") as check_file:
                        for chunk in iter(lambda: check_file.read(128 * 1024), b""):
                            check.update(chunk)
                    if check.hexdigest() != sha:
                        raise IOError("Google Drive copy verification failed (SHA-256 mismatch)")
                    os.replace(drive_partial, destination)
                finally:
                    drive_partial.unlink(missing_ok=True)
                # Record success ONLY after the complete file exists in Drive.
                record_result(row, "downloaded", saved_path=str(destination), digest=sha)
                known_hashes.add(sha)
                successful_urls.add(row["file_url"])
                stats["downloaded"] += 1
                logger.info("Downloaded %s (%s bytes)", destination.name, total_size)
        except Exception as exc:
            stats["failed"] += 1
            record_result(row, "failed", error=str(exc))
            logger.error("Download failed: %s | %s", row["filename"], exc)
        finally:
            temporary.unlink(missing_ok=True)

    print("Download results:", stats)
    print("Saved PDFs:", PDF_DIR)
    print("Persistent manifest:", MANIFEST_PATH)

    run_summary = {
        "finished_at_colombia": datetime.now(TIMEZONE).isoformat(),
        "output_directory": str(OUTPUT_DIR),
        "repository_count": len(repositories),
        "pdf_candidates_on_first_pages": len(candidates),
        "pdfs_selected_this_run": len(selected),
        "first_run": initial_run,
        "results": stats,
        "discovery_errors": discovery_errors,
        "manifest_rows": len(manifest),
    }
    with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
        json.dump(run_summary, f, ensure_ascii=False, indent=2)
    logger.info("Run summary saved to %s", SUMMARY_PATH)
    print(json.dumps(run_summary, ensure_ascii=False, indent=2))
    display(manifest.tail(20))

    # ----- Retención: 7 días o publicación más reciente de la serie -----
    def urls_vencidas(filas):
        """Devuelve las URL que ya no se conservan."""
        corte = (TODAY - timedelta(days=DIAS_RETENCION)).isoformat()
        activas = filas[filas["status"].isin(["downloaded", "duplicate_content"])]
        conservar = set()

        # Cada periodo se compara dentro de su repositorio: un boletín
        # diario no reemplaza a uno mensual.
        for (_, precision), grupo in activas.groupby(["category", "date_precision"], sort=False):
            if precision in ("day", "month"):
                vigentes = grupo["file_date"] == grupo["file_date"].max()
                if precision == "day":
                    vigentes = vigentes | (grupo["file_date"] >= corte)
            else:
                vigentes = grupo["downloaded_at"] == grupo["downloaded_at"].max()
            conservar.update(grupo.loc[vigentes, "file_url"])

        # Si un conservado es copia de otro archivo, ese archivo también se queda.
        hashes = set(activas.loc[activas["file_url"].isin(conservar), "sha256"]) - {""}
        conservar.update(activas.loc[activas["sha256"].isin(hashes), "file_url"])
        return set(activas["file_url"]) - conservar

    PERIODOS = {"day": "diario o semanal", "month": "mensual"}
    MOTIVO = f"Más de {DIAS_RETENCION} días y ya no es lo último de su serie"
    errores_borrado = 0
    vencidas = urls_vencidas(manifest)

    for indice, fila in manifest.iterrows():
        archivo = Path(fila["saved_path"]).name if fila["saved_path"] else fila["filename"]
        datos_fila = dict(
            fecha=fila["file_date"],
            periodo=PERIODOS.get(fila["date_precision"], "sin fecha en el nombre"),
        )
        if fila["status"] == "deleted":
            registrar("Boletines", fila["category"], archivo, estado="borrado", motivo=MOTIVO, **datos_fila)
        elif fila["status"] == "failed":
            registrar("Boletines", fila["category"], archivo, estado="error", motivo=fila["error"][:200], **datos_fila)
        elif fila["file_url"] not in vencidas:
            registrar("Boletines", fila["category"], archivo, **datos_fila)
        elif not BORRAR_ARCHIVOS:
            registrar("Boletines", fila["category"], archivo, estado="por_borrar", motivo=MOTIVO, **datos_fila)
        else:
            try:
                if fila["status"] == "downloaded" and fila["saved_path"]:
                    (PDF_DIR / archivo).unlink(missing_ok=True)
                manifest.loc[indice, "status"] = "deleted"
                manifest.loc[indice, "deleted_at"] = datetime.now(TIMEZONE).isoformat()
                registrar("Boletines", fila["category"], archivo, estado="borrado", motivo=MOTIVO, **datos_fila)
            except OSError as exc:
                errores_borrado += 1
                logger.error("Could not delete %s: %s", archivo, exc)

    write_manifest()
    print(f"Boletines vencidos: {len(vencidas)} | "
          f"{'borrados' if BORRAR_ARCHIVOS else 'solo marcados (simulación)'}")

    return {
        "repositorios": len(repositories),
        "pdf_descargados": stats["downloaded"],
        "pdf_con_error": stats["failed"],
        "boletines_vencidos": len(vencidas),
        "avisos": stats["failed"] + len(discovery_errors) + errores_borrado,
    }


## 5. Módulo 3: Monitor de la Cuenca Amazónica

Genera un PDF por evento, en una carpeta por país, a partir del repositorio en GitHub. Un evento sin cambios no genera un archivo nuevo; si cambia, sale otro PDF solo de ese evento. No consulta al IDEAM. Parámetros principales: `SOLO_ACTIVOS`, `PAISES` y `EVENTOS_EXCLUIDOS`.

In [ ]:
# 5. Módulo 3: Monitor de la Cuenca Amazónica
def ejecutar_monitor():
    # 2. Configuración
    REPO = "3is-global/echo_dp_amazon_monitor"
    RAMA = "main"
    SOLO_ACTIVOS = False   # True: solo eventos "En curso" y "Pronóstico"

    # Países del proyecto (clave = valor de COUNTRY en Events/, valor = nombre en el PDF/carpeta).
    # Se genera un PDF para cada uno aunque no tenga eventos.
    PAISES = {
        "Bolivia": "Bolivia",
        "Brazil": "Brasil",
        "Colombia": "Colombia",
        "Ecuador": "Ecuador",
        "Peru": "Perú",
        "Venezuela": "Venezuela",
    }

    BASE_SALIDA = DIR_MONITOR

    # Eventos que no se publican. AMZ-20260916-003 es un pronóstico que proviene del IDEAM.
    EVENTOS_EXCLUIDOS = {"AMZ-20260916-003"}


    # 3. Descarga de datos (eventos del repositorio)
    import json
    import sys
    import time
    from datetime import datetime, timezone
    from zoneinfo import ZoneInfo

    import requests

    HEADERS = {"User-Agent": "amazon-basin-monitor-pdf/1.0 (+https://github.com/3is-global/echo_dp_amazon_monitor)"}
    SESSION = requests.Session()
    SESSION.headers.update(HEADERS)

    RAW_BASE = f"https://raw.githubusercontent.com/{REPO}/{RAMA}"
    API_BASE = f"https://api.github.com/repos/{REPO}/contents"


    def http_get(url, retries=3, timeout=30):
        ultimo_error = None
        for intento in range(1, retries + 1):
            try:
                r = SESSION.get(url, timeout=timeout)
                r.raise_for_status()
                return r.content
            except Exception as e:  # noqa: BLE001
                ultimo_error = e
                print(f"  [aviso] intento {intento}/{retries} falló para {url}: {e}", file=sys.stderr)
                time.sleep(2 * intento)
        raise RuntimeError(f"No se pudo descargar {url}: {ultimo_error}")


    def leer_json(url):
        return json.loads(http_get(url).decode("utf-8"))


    def listar_json_github(carpeta):
        """Lista los .json de una carpeta del repo vía la API de GitHub."""
        items = leer_json(f"{API_BASE}/{carpeta}?ref={RAMA}")
        return [it for it in items if it.get("type") == "file" and it["name"].endswith(".json")]


    def cargar_eventos():
        try:
            archivos = listar_json_github("Events")
            eventos = [leer_json(it["download_url"]) for it in archivos]
            origen = f"GitHub {REPO} (Events/, {len(archivos)} archivos)"
        except Exception as e:  # noqa: BLE001
            print(f"[aviso] No se pudo leer Events/ archivo por archivo ({e}); uso events.json", file=sys.stderr)
            eventos = leer_json(f"{RAW_BASE}/events.json")
            origen = f"GitHub {REPO} (events.json)"
        return eventos, origen


    def cargar_impactos():
        """Cifras de impacto consolidadas (Impacts/<EVENT_ID>.json). Opcional."""
        try:
            return {it["name"][:-5]: leer_json(it["download_url"]) for it in listar_json_github("Impacts")}
        except Exception as e:  # noqa: BLE001
            print(f"[aviso] No se pudieron leer los impactos consolidados: {e}", file=sys.stderr)
            return {}

    # 4. Generación de los PDF
    from xml.sax.saxutils import escape

    from reportlab.lib import colors
    from reportlab.lib.enums import TA_LEFT
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import cm
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.platypus import (KeepTogether, PageBreak, Paragraph, SimpleDocTemplate,
                                    Spacer, Table, TableStyle)


    def registrar_fuente():
        """DejaVu Sans (incluida en matplotlib) cubre acentos, ñ, ç, ã, ≈, etc."""
        try:
            import matplotlib
            carpeta = Path(matplotlib.get_data_path()) / "fonts" / "ttf"
            pdfmetrics.registerFont(TTFont("DejaVu", str(carpeta / "DejaVuSans.ttf")))
            pdfmetrics.registerFont(TTFont("DejaVu-Bold", str(carpeta / "DejaVuSans-Bold.ttf")))
            pdfmetrics.registerFontFamily("DejaVu", normal="DejaVu", bold="DejaVu-Bold",
                                          italic="DejaVu", boldItalic="DejaVu-Bold")
            return "DejaVu", "DejaVu-Bold"
        except Exception as e:  # noqa: BLE001
            print(f"[aviso] Sin DejaVu Sans ({e}); uso Helvetica", file=sys.stderr)
            return "Helvetica", "Helvetica-Bold"


    FUENTE, FUENTE_NEGRITA = registrar_fuente()

    base = getSampleStyleSheet()
    E = {
        "titulo": ParagraphStyle("titulo", parent=base["Title"], fontName=FUENTE_NEGRITA, fontSize=18,
                                 alignment=TA_LEFT, spaceAfter=4, textColor=colors.HexColor("#1b4332")),
        "sub": ParagraphStyle("sub", parent=base["Normal"], fontName=FUENTE, fontSize=9,
                              textColor=colors.HexColor("#555555"), spaceAfter=10),
        "h1": ParagraphStyle("h1", parent=base["Heading1"], fontName=FUENTE_NEGRITA, fontSize=14,
                             spaceBefore=12, spaceAfter=6, textColor=colors.HexColor("#1b4332")),
        "h2": ParagraphStyle("h2", parent=base["Heading2"], fontName=FUENTE_NEGRITA, fontSize=11.5,
                             spaceBefore=10, spaceAfter=4),
        "h3": ParagraphStyle("h3", parent=base["Heading3"], fontName=FUENTE_NEGRITA, fontSize=10,
                             spaceBefore=6, spaceAfter=2),
        "normal": ParagraphStyle("normal", parent=base["Normal"], fontName=FUENTE, fontSize=9, leading=12),
        "nota": ParagraphStyle("nota", parent=base["Normal"], fontName=FUENTE, fontSize=8, leading=10.5,
                               textColor=colors.HexColor("#555555")),
        "celda": ParagraphStyle("celda", parent=base["Normal"], fontName=FUENTE, fontSize=8, leading=10),
        "celda_b": ParagraphStyle("celda_b", parent=base["Normal"], fontName=FUENTE_NEGRITA, fontSize=8,
                                  leading=10, textColor=colors.white),
    }

    ESTADO = {"ONGOING": "En curso", "FORECAST": "Pronóstico", "OBSERVED": "Observado", "ENDED": "Finalizado"}
    PRIORIDAD = {"RED": "Roja", "ORANGE": "Naranja", "YELLOW": "Amarilla", "GREEN": "Verde"}
    COLOR_PRIORIDAD = {"RED": "#c1121f", "ORANGE": "#f77f00", "YELLOW": "#fcbf49", "GREEN": "#2d6a4f"}
    ORDEN_PRIORIDAD = {"RED": 0, "ORANGE": 1, "YELLOW": 2, "GREEN": 3}
    AMENAZA = {
        "Drought": "Sequía", "Earthquake": "Sismo", "Wildfire": "Incendio forestal",
        "Extreme Rain": "Lluvia extrema", "Flood": "Inundación", "Landslide": "Deslizamiento",
        "Deforestation": "Deforestación", "Extreme Heat": "Calor extremo", "Storm": "Tormenta",
    }
    NIVEL = {"HIGH": "Alta", "MEDIUM": "Media", "LOW": "Baja"}
    CUENCA = {"INSIDE": "Dentro", "PARTIALLY_INSIDE": "Parcialmente dentro", "OUTSIDE": "Fuera"}
    VACIOS = {"", "NOT_REPORTED", "UNAVAILABLE", "NONE", "NULL", "N/A"}


    def p(texto, estilo="normal"):
        return Paragraph(escape(str(texto)).replace("\n", "<br/>"), E[estilo])


    def valor(v):
        """Texto legible; None si el campo está vacío o 'NOT_REPORTED'."""
        if v is None:
            return None
        s = str(v).strip()
        if s.upper() in VACIOS:
            return None
        return s.replace("NOT_REPORTED", "No reportado").replace("UNAVAILABLE", "No disponible")


    def traducir(tabla, v):
        return tabla.get(v, str(v or "").replace("_", " ").capitalize() or "—")


    def enlace(url):
        u = escape(url.strip())
        return f'<link href="{u}" color="#1d4e89">{u}</link>'


    def tabla(datos, anchos, encabezado=True, color_fila=None):
        t = Table(datos, colWidths=anchos, repeatRows=1 if encabezado else 0)
        estilo = [
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("GRID", (0, 0), (-1, -1), 0.3, colors.HexColor("#bbbbbb")),
            ("LEFTPADDING", (0, 0), (-1, -1), 4), ("RIGHTPADDING", (0, 0), (-1, -1), 4),
            ("TOPPADDING", (0, 0), (-1, -1), 3), ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
        ]
        if encabezado:
            estilo.append(("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1b4332")))
        for (fila, col), color in (color_fila or {}).items():
            estilo.append(("BACKGROUND", (col, fila), (col, fila), colors.HexColor(color)))
        t.setStyle(TableStyle(estilo))
        return t


    def encabezados(*titulos):
        return [Paragraph(escape(x), E["celda_b"]) for x in titulos]


    def celda_color(texto):
        """Celda con texto blanco en negrita (para fondos de color)."""
        return Paragraph(escape(texto), E["celda_b"])


    def bloque_evento(ev, impacto):
        titulo = ev.get("EVENT_TITLE_ES") or ev.get("EVENT_TITLE_EN") or ""
        out = [p(f"{ev.get('EVENT_ID', '')} — {titulo}", "h2")]

        ubicacion = " · ".join(x for x in (valor(ev.get("ADMINISTRATIVE_LEVEL_1")), valor(ev.get("ADMINISTRATIVE_LEVEL_2")),
                                            valor(ev.get("MUNICIPALITY")), valor(ev.get("LOCALITY")),
                                            valor(ev.get("TERRITORY"))) if x)
        coords = ""
        if valor(ev.get("LATITUDE")) and valor(ev.get("LONGITUDE")):
            coords = f"{ev['LATITUDE']}, {ev['LONGITUDE']} ({traducir({'APPROXIMATE': 'aproximadas', 'EXACT': 'exactas'}, ev.get('COORDINATE_ACCURACY')).lower()})"
        fechas = f"Inicio: {valor(ev.get('EVENT_START_DATE')) or 'No reportado'} · Fin: {valor(ev.get('EVENT_END_DATE')) or 'No reportado'}"
        ficha = [
            ("Amenaza", f"{traducir(AMENAZA, ev.get('HAZARD_TYPE'))}" + (f" — {ev['HAZARD_SUBTYPE']}" if valor(ev.get("HAZARD_SUBTYPE")) else "")),
            ("Estado / prioridad", f"{traducir(ESTADO, ev.get('EVENT_STATUS'))} · prioridad {traducir(PRIORIDAD, ev.get('ALERT_PRIORITY')).lower()} · relevancia AA {traducir(NIVEL, ev.get('AA_RELEVANCE')).lower()}"),
            ("Ubicación", ubicacion or "No reportada"),
            ("Coordenadas", coords or "No reportadas"),
            ("Cuenca amazónica", traducir(CUENCA, ev.get("AMAZON_BASIN_STATUS"))),
            ("Fechas", fechas),
            ("Detección / actualización", f"Detectado {ev.get('FIRST_DETECTED_DATE', '—')} · actualizado {ev.get('LAST_UPDATED_DATE', '—')} ({ev.get('UPDATE_COUNT', '0')} actualizaciones)"),
            ("Verificación / confianza", f"{traducir({}, ev.get('VERIFICATION_STATUS'))} · confianza {traducir(NIVEL, ev.get('CONFIDENCE')).lower()}"),
        ]
        out.append(tabla([[p(k, "celda"), p(v, "celda")] for k, v in ficha], [4.2 * cm, 13.5 * cm], encabezado=False))

        desc = valor(ev.get("EVENT_DESCRIPTION_ES")) or valor(ev.get("EVENT_DESCRIPTION_EN"))
        if desc:
            out += [p("Descripción", "h3"), p(desc)]

        impactos = [(etq, valor(ev.get(campo))) for etq, campo in (
            ("Muertes", "DEATHS"), ("Heridos", "INJURIES"), ("Población afectada", "AFFECTED_POPULATION"),
            ("Desplazados", "DISPLACED_POPULATION"), ("Evacuados", "EVACUATED_POPULATION"),
            ("Agricultura", "AGRICULTURAL_IMPACT"), ("Ganadería", "LIVESTOCK_IMPACT"), ("Bosques", "FOREST_IMPACT"),
            ("Agua", "WATER_IMPACT"), ("Seguridad alimentaria", "FOOD_SECURITY_IMPACT"))]
        impactos = [(k, v) for k, v in impactos if v]
        if impactos:
            out += [p("Impactos reportados", "h3"),
                    tabla([[p(k, "celda"), p(v, "celda")] for k, v in impactos], [4.2 * cm, 13.5 * cm], encabezado=False)]

        if impacto:
            cifras = [(etq, impacto.get(campo)) for etq, campo in (
                ("Muertes", "DEATHS"), ("Heridos", "INJURIES"), ("Personas afectadas", "AFFECTED_PERSONS"),
                ("Familias afectadas", "AFFECTED_FAMILIES"), ("Desplazados", "DISPLACED_PERSONS"),
                ("Evacuados", "EVACUATED_PERSONS"))]
            cifras = [(k, f"{v:,}".replace(",", ".") if isinstance(v, (int, float)) else str(v)) for k, v in cifras if v is not None]
            out.append(p(f"Cifras consolidadas (al {impacto.get('AS_OF_DATE', '—')})", "h3"))
            if cifras:
                out.append(tabla([[p(k, "celda"), p(v, "celda")] for k, v in cifras], [4.2 * cm, 13.5 * cm], encabezado=False))
            if valor(impacto.get("NOTES")):
                out.append(p(impacto["NOTES"], "nota"))
            if valor(impacto.get("SOURCE")):
                fuente = escape(impacto["SOURCE"])
                if valor(impacto.get("SOURCE_URL")):
                    fuente += " — " + enlace(impacto["SOURCE_URL"])
                out.append(Paragraph("Fuente: " + fuente, E["nota"]))

        notas = valor(ev.get("NOTES_ES")) or valor(ev.get("NOTES_EN"))
        if notas:
            out += [p("Notas", "h3"), p(notas, "nota")]

        fuentes = []
        for nombre, url in (("PRIMARY_SOURCE", "PRIMARY_SOURCE_URL"), ("SECONDARY_SOURCE", "SECONDARY_SOURCE_URL")):
            if valor(ev.get(nombre)) or valor(ev.get(url)):
                txt = escape(valor(ev.get(nombre)) or "")
                if valor(ev.get(url)):
                    txt += (" — " if txt else "") + enlace(ev[url])
                fuentes.append(txt)
        if valor(ev.get("GDACS_URL")):
            fuentes.append("GDACS — " + enlace(ev["GDACS_URL"]))
        for otra in (valor(ev.get("OTHER_SOURCE_URLS")) or "").split(";"):
            otra = otra.strip()
            if not otra:
                continue
            if otra.startswith("http"):
                url, _, resto = otra.partition(" ")
                fuentes.append(enlace(url) + (" " + escape(resto) if resto else ""))
            else:
                fuentes.append(escape(otra))
        if fuentes:
            out.append(p("Fuentes", "h3"))
            out += [Paragraph("• " + f, E["nota"]) for f in fuentes]
        out.append(Spacer(1, 8))
        return out


    def generar_pdf(ruta, nombre_pais, ev, impacto, origen_eventos, ahora):
        ruta.parent.mkdir(parents=True, exist_ok=True)
        tmp = ruta.with_suffix(".tmp.pdf")

        def pie(canvas, doc):
            canvas.saveState()
            canvas.setFont(FUENTE, 7.5)
            canvas.setFillColor(colors.HexColor("#777777"))
            canvas.drawString(1.5 * cm, 1.0 * cm, f"Monitor de la Cuenca Amazónica · {nombre_pais} · {ahora:%Y-%m-%d %H:%M} (hora Bogotá)")
            canvas.drawRightString(A4[0] - 1.5 * cm, 1.0 * cm, f"Página {doc.page}")
            canvas.restoreState()

        doc = SimpleDocTemplate(str(tmp), pagesize=A4, leftMargin=1.5 * cm, rightMargin=1.5 * cm,
                                topMargin=1.5 * cm, bottomMargin=1.6 * cm,
                                title=f"Alerta {nombre_pais} {ev.get('EVENT_ID', '')}", author="Monitor de la Cuenca Amazónica")
        hist = [
            p(f"Alertas — {nombre_pais}", "titulo"),
            p(f"Monitor de la Cuenca Amazónica · generado el {ahora:%Y-%m-%d %H:%M} (hora Bogotá) · "
              f"eventos: {origen_eventos}" + (" · solo eventos en curso y pronósticos" if SOLO_ACTIVOS else ""), "sub"),
            *bloque_evento(ev, impacto),
        ]
        doc.build(hist, onFirstPage=pie, onLaterPages=pie)
        tmp.replace(ruta)  # reemplazo atómico: nunca queda un PDF a medio escribir

    # 5. Ejecutar: descargar, generar y guardar en Drive
    ahora = datetime.now(ZoneInfo("America/Bogota"))
    fecha = ahora.strftime("%Y-%m-%d")

    print("Descargando eventos del repositorio...")
    eventos, origen_eventos = cargar_eventos()
    impactos = cargar_impactos()
    eventos = [e for e in eventos if e.get("EVENT_ID") not in EVENTOS_EXCLUIDOS]
    print(f"  -> {len(eventos)} eventos, {len(impactos)} con cifras consolidadas")

    if SOLO_ACTIVOS:
        eventos = [e for e in eventos if e.get("EVENT_STATUS") in ("ONGOING", "FORECAST")]

    desconocidos = sorted({e.get("COUNTRY") for e in eventos} - set(PAISES))
    if desconocidos:
        print(f"[aviso] Países sin entrada en PAISES (se generan igual): {desconocidos}")
    paises = {**PAISES, **{c: c for c in desconocidos if c}}

    pdf_nuevos = 0
    pdf_sin_cambios = 0

    for clave, nombre in paises.items():
        carpeta_pais = BASE_SALIDA / nombre
        del_pais = [e for e in eventos if e.get("COUNTRY") == clave]

        for ev in del_pais:
            id_evento = ev.get("EVENT_ID") or "sin_id"
            serie = f"{nombre} / {id_evento}"
            impacto = impactos.get(ev.get("EVENT_ID"))
            huella = hashlib.sha256(json.dumps(
                [ev, impacto], sort_keys=True, ensure_ascii=False, default=str
            ).encode("utf-8")).hexdigest()

            # Un evento sin cambios conserva su PDF, sin generar otro archivo.
            anterior = next((
                f for f in REGISTRO.values()
                if f["modulo"] == "Monitor" and f["serie"] == serie
                and f["estado"] == "vigente" and f["huella"] == huella
                and (carpeta_pais / f["archivo"]).exists()
            ), None)

            if anterior:
                ruta = carpeta_pais / anterior["archivo"]
                pdf_sin_cambios += 1
            else:
                ruta = carpeta_pais / f"{nombre}_{id_evento}_{fecha}.pdf"
                # Si el evento cambió el mismo día, otro archivo con la hora.
                if ruta.exists():
                    ruta = ruta.with_name(
                        f"{nombre}_{id_evento}_{fecha}_obtenido_{ahora:%H%M%S}.pdf"
                    )
                generar_pdf(ruta, nombre, ev, impacto, origen_eventos, ahora)
                pdf_nuevos += 1

            registrar("Monitor", serie, ruta.name,
                      fecha=anterior["fecha_archivo"] if anterior else fecha,
                      periodo="última versión por evento", huella=huella)

            # En Drive solo se conserva la última versión del evento.
            for viejo in carpeta_pais.glob(f"{nombre}_{id_evento}_*.pdf"):
                if viejo == ruta:
                    continue
                motivo = "Reemplazado por una versión más reciente del evento"
                if BORRAR_ARCHIVOS:
                    viejo.unlink(missing_ok=True)
                    registrar("Monitor", serie, viejo.name, estado="borrado", motivo=motivo)
                else:
                    registrar("Monitor", serie, viejo.name, estado="por_borrar", motivo=motivo)

        print(f"  {nombre:<10} {len(del_pais):>3} evento(s)")

    print(f"PDF nuevos: {pdf_nuevos} | sin cambios: {pdf_sin_cambios}")

    return {
        "eventos": len(eventos),
        "pdf_nuevos": pdf_nuevos,
        "pdf_sin_cambios": pdf_sin_cambios,
        "avisos": 0,
    }


## 6. Módulo 4: INAMHI El Niño

Descarga el último informe publicado en `servicios.inamhi.gob.ec/fenomeno-el-nino/` (botón **Descargar** del "Último informe localizado"; hoy es el informe ERFEN, alojado en `inocar.mil.ec`). Lo guarda siempre con el mismo nombre, `inamhi_elnino/inamhi_elnino_ultimo_informe.pdf`, así que cada informe nuevo **sobrescribe** al anterior. Si el PDF no cambió, el archivo no se toca.

In [ ]:
# 6. Módulo 4: INAMHI El Niño
def ejecutar_inamhi_elnino():
    import re
    from urllib.parse import urljoin, urlparse

    import requests
    from bs4 import BeautifulSoup

    PAGINA = "https://servicios.inamhi.gob.ec/fenomeno-el-nino/"
    NOMBRE_PDF = "inamhi_elnino_ultimo_informe.pdf"  # Nombre fijo: se sobrescribe
    DESTINO = DIR_INAMHI / NOMBRE_PDF

    sesion = requests.Session()
    sesion.headers.update({
        "User-Agent": "webscraping-anticipa/1.0 (+https://github.com/lreiners-3is/webscraping-anticipa)"
    })

    def obtener(url, timeout=60, intentos=3):
        ultimo_error = None
        for intento in range(1, intentos + 1):
            try:
                r = sesion.get(url, timeout=timeout)
                r.raise_for_status()
                return r
            except requests.RequestException as e:
                ultimo_error = e
                print(f"  [aviso] intento {intento}/{intentos} falló para {url}: {e}")
                time.sleep(3 * intento)
        raise RuntimeError(f"No se pudo descargar {url}: {ultimo_error}")

    # Enlace "Descargar" del último informe; si cambia el diseño de la página,
    # se usa el primer PDF del panel de seguimiento.
    sopa = BeautifulSoup(obtener(PAGINA).text, "html.parser")
    enlace = sopa.select_one("a[data-enino-latest-download]")
    if enlace is None:
        panel = sopa.select_one("#enino-panel-reports")
        if panel is not None:
            enlace = panel.find("a", href=re.compile(r"\.pdf($|[?#])", re.I))
    if enlace is None or not enlace.get("href"):
        raise RuntimeError("No se encontró el enlace al último informe en la página")

    url_pdf = urljoin(PAGINA, enlace["href"]).split("#")[0]
    meta = sopa.select_one("[data-enino-latest-meta]")
    informe = " ".join(meta.get_text(" ").split()) if meta else "Último informe"

    contenido = obtener(url_pdf, timeout=120).content
    if not contenido.startswith(b"%PDF-"):
        raise ValueError(f"La respuesta de {url_pdf} no es un PDF")

    huella = hashlib.sha256(contenido).hexdigest()
    anterior = hashlib.sha256(DESTINO.read_bytes()).hexdigest() if DESTINO.exists() else None

    if huella == anterior:
        cambio = "sin cambios"
    else:
        # Se escribe aparte y se reemplaza de una vez, para no dejar un PDF a medias.
        temporal = DESTINO.with_name(DESTINO.name + ".part")
        try:
            temporal.write_bytes(contenido)
            os.replace(temporal, DESTINO)
        finally:
            temporal.unlink(missing_ok=True)
        cambio = "nuevo" if anterior is None else "sobrescrito"

    fecha = re.search(r"(20\d{2})(\d{2})(\d{2})", Path(urlparse(url_pdf).path).name)
    registrar(
        "INAMHI El Niño", informe, NOMBRE_PDF,
        fecha="-".join(fecha.groups()) if fecha else "",
        periodo="se sobrescribe en cada corrida",
        motivo=f"Fuente: {url_pdf}",
        huella=huella,
    )

    print(f"{informe}\nFuente: {url_pdf}\nPDF: {DESTINO} ({len(contenido):,} bytes) -> {cambio}")
    return {"informe": informe, "pdf": cambio, "avisos": 0}

## 7. Módulo 5: INAMHI Boletines

Descarga los boletines publicados en `servicios.inamhi.gob.ec/boletines/` (hoy son 9). Cada boletín tiene una URL fija cuyo contenido el INAMHI reemplaza, así que:

- Antes de descargar se consulta la fecha (`Last-Modified`) y la huella (`ETag`) del servidor. Si no cambiaron, no se descarga.
- Si cambiaron pero el contenido es idéntico (SHA-256), no se guarda otra copia.
- Cada versión nueva se guarda como `<Boletín>_AAAA-MM-DD.pdf`, con la fecha de publicación en el servidor (hora de Ecuador). Si hay dos versiones el mismo día, la segunda lleva `_HHMMSS`.

Nada se sobrescribe ni se borra. El control está en `registro_archivos.csv`: la huella va en `huella`, y la URL, `Last-Modified` y `ETag` en `motivo`.

In [ ]:
# 7. Módulo 5: INAMHI Boletines
def ejecutar_inamhi_boletines():
    import re
    import unicodedata
    from email.utils import parsedate_to_datetime
    from urllib.parse import urljoin

    import requests
    from bs4 import BeautifulSoup

    PAGINA = "https://servicios.inamhi.gob.ec/boletines/"
    PAUSA_SEGUNDOS = 1.0
    TZ_ECUADOR = ZoneInfo("America/Guayaquil")
    MODULO = "INAMHI Boletines"
    PERIODO = "se conserva siempre"

    # El control vive en registro_archivos.csv; el manifest propio ya no se usa.
    (DIR_INAMHI_BOLETINES / "manifest_inamhi_boletines.csv").unlink(missing_ok=True)

    sesion = requests.Session()
    sesion.headers.update({
        "User-Agent": "webscraping-anticipa/1.0 (+https://github.com/lreiners-3is/webscraping-anticipa)"
    })

    def pedir(metodo, url, timeout=60, intentos=3):
        ultimo_error = None
        for intento in range(1, intentos + 1):
            try:
                r = sesion.request(metodo, url, timeout=timeout, allow_redirects=True)
                r.raise_for_status()
                return r
            except requests.RequestException as e:
                ultimo_error = e
                print(f"  [aviso] intento {intento}/{intentos} falló para {url}: {e}")
                time.sleep(3 * intento)
        raise RuntimeError(f"No se pudo consultar {url}: {ultimo_error}")

    def nombre_seguro(texto):
        texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
        return re.sub(r"[^A-Za-z0-9]+", "_", texto).strip("_")[:80] or "boletin"

    def motivo(url, last_modified, etag):
        return f"Fuente: {url} | Last-Modified: {last_modified} | ETag: {etag}"

    def leer_motivo(texto):
        m = re.match(r"Fuente: (\S+)(?: \| Last-Modified: (.*?) \| ETag: (.*))?$", texto or "")
        return (m.group(1), m.group(2) or "", m.group(3) or "") if m else ("", "", "")

    def filas_del_modulo():
        return [f for f in REGISTRO.values() if f["modulo"] == MODULO and f["estado"] == "vigente"]

    def ultima_de(url):
        """Fila más reciente del registro para esa URL."""
        propias = [f for f in filas_del_modulo() if leer_motivo(f["motivo"])[0] == url]
        return max(propias, key=lambda f: f["primera_vez"]) if propias else None

    _huellas_carpeta = {}

    def huellas_en_carpeta():
        """SHA-256 de los PDF que ya están en la carpeta (se calcula una sola vez)."""
        if not _huellas_carpeta:
            for pdf in DIR_INAMHI_BOLETINES.glob("*.pdf"):
                _huellas_carpeta[hashlib.sha256(pdf.read_bytes()).hexdigest()] = pdf.name
        return _huellas_carpeta

    def volver_a_registrar(fila, nuevo_motivo=None):
        registrar(MODULO, fila["serie"], fila["archivo"], fecha=fila["fecha_archivo"],
                  periodo=PERIODO, motivo=nuevo_motivo or fila["motivo"], huella=fila["huella"])

    # ----- Boletines publicados en la página -----
    sopa = BeautifulSoup(pedir("GET", PAGINA).text, "html.parser")
    boletines = {}
    for enlace in sopa.select("a.ihb-read[data-pdf-source]"):
        url = urljoin(PAGINA, enlace["data-pdf-source"]).split("#")[0]
        tarjeta = enlace.find_parent("article")
        titulo = tarjeta.find("h3") if tarjeta else None
        boletines[url] = " ".join(titulo.get_text(" ").split()) if titulo else Path(url).stem
    if not boletines:
        raise RuntimeError("No se encontraron boletines en la página")

    stats = {"boletines_en_pagina": len(boletines), "pdf_nuevos": 0, "sin_cambios": 0, "pdf_con_error": 0}

    for url, boletin in boletines.items():
        try:
            time.sleep(PAUSA_SEGUNDOS)
            cabecera = pedir("HEAD", url).headers
            last_modified = cabecera.get("Last-Modified", "")
            etag = cabecera.get("ETag", "")
            anterior = ultima_de(url)

            # 1. El servidor dice que no cambió: no se descarga.
            if anterior and (etag or last_modified) and \
                    leer_motivo(anterior["motivo"])[1:] == (last_modified, etag):
                stats["sin_cambios"] += 1
                print(f"  sin cambios: {boletin}")
                continue

            time.sleep(PAUSA_SEGUNDOS)
            contenido = pedir("GET", url, timeout=180).content
            if not contenido.startswith(b"%PDF-"):
                raise ValueError("La respuesta no es un PDF")
            sha = hashlib.sha256(contenido).hexdigest()

            # 2. Cambió la cabecera pero no el contenido: no se guarda otra copia.
            #    Se anota la cabecera nueva para no volver a descargarlo.
            igual = next((f for f in filas_del_modulo() if f["huella"] == sha), None)
            if igual:
                volver_a_registrar(igual, motivo(url, last_modified, etag))
                stats["sin_cambios"] += 1
                print(f"  mismo contenido: {boletin}")
                continue

            # Respaldo: si el PDF ya está en la carpeta pero no en el registro,
            # se registra ese archivo en lugar de guardar otra copia.
            en_carpeta = huellas_en_carpeta().get(sha)
            if en_carpeta:
                fecha_archivo = re.search(r"_(\d{4}-\d{2}-\d{2})", en_carpeta)
                registrar(MODULO, boletin, en_carpeta,
                          fecha=fecha_archivo.group(1) if fecha_archivo else "",
                          periodo=PERIODO, motivo=motivo(url, last_modified, etag), huella=sha)
                stats["sin_cambios"] += 1
                print(f"  ya estaba en la carpeta: {en_carpeta}")
                continue

            # 3. Versión nueva: archivo nuevo con la fecha de publicación.
            if last_modified:
                publicado = parsedate_to_datetime(last_modified).astimezone(TZ_ECUADOR)
            else:
                publicado = datetime.now(TZ_ECUADOR)
            base = f"{nombre_seguro(boletin)}_{publicado:%Y-%m-%d}"
            destino = DIR_INAMHI_BOLETINES / f"{base}.pdf"
            if destino.exists():
                destino = DIR_INAMHI_BOLETINES / f"{base}_{publicado:%H%M%S}.pdf"
            if destino.exists():
                raise FileExistsError(f"Ya existe {destino.name} con otro contenido")
            temporal = destino.with_name(destino.name + ".part")
            try:
                temporal.write_bytes(contenido)
                os.replace(temporal, destino)
            finally:
                temporal.unlink(missing_ok=True)

            registrar(MODULO, boletin, destino.name, fecha=publicado.date().isoformat(),
                      periodo=PERIODO, motivo=motivo(url, last_modified, etag), huella=sha)
            stats["pdf_nuevos"] += 1
            print(f"  NUEVO: {destino.name} ({len(contenido):,} bytes)")
        except Exception as e:  # Un boletín con error no detiene a los demás
            stats["pdf_con_error"] += 1
            print(f"  [error] {boletin}: {e}")

    # Los PDF ya guardados siguen vigentes: se actualiza su fecha de revisión.
    for fila in filas_del_modulo():
        volver_a_registrar(fila)

    print(f"Boletines en la página: {stats['boletines_en_pagina']} | nuevos: {stats['pdf_nuevos']} | "
          f"sin cambios: {stats['sin_cambios']} | con error: {stats['pdf_con_error']}")
    return {**stats, "avisos": stats["pdf_con_error"]}

## 8. Módulo 6: INAMHI Pronóstico

Pronóstico del tiempo por provincia y ciudad de `servicios.inamhi.gob.ec/pronostico/`. Los valores salen del servicio que alimenta el mapa de esa página (`/wp-json/inamhi/v1/banner`): por cada ciudad, temperatura mínima y máxima (°C), índice UV, lluvia y la condición por período del día. Solo se toma el pronóstico de hoy, el mismo que muestra el mapa (lista `today`; si una ciudad aún tiene la fecha de ayer y `next` ya trae la de hoy, se usa esa, como hace el mapa).

- **Fidelidad:** solo se copian los valores que publica la fuente. Lo que no viene queda en `null`. Si la fuente indica que una ciudad usa el pronóstico de otra (`derivada`, `ciudad_base`), se conserva esa marca. Una provincia sin ciudades en la fuente aparece sin valores: no se le asigna ningún pronóstico.
- **Boletines regionales:** los PDF por zona que enlaza el mapa se listan con su URL y las provincias que cubren, pero no se extraen valores de ellos: son infografías y su texto no permite asociar con seguridad cada valor a un lugar.
- **Archivos:** en `inamhi_pronostico_diario/`, `inamhi_pronostico_AAAA-MM-DD.json` y `.pdf`, con la fecha del pronóstico que informa la fuente. Las corridas del mismo día los sobrescriben solo si los datos cambiaron. Al llegar un día nuevo se borran los del día anterior.

In [ ]:
# 8. Módulo 6: INAMHI Pronóstico
def ejecutar_inamhi_pronostico():
    import re
    from email.utils import parsedate_to_datetime

    import requests
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import cm
    from reportlab.platypus import (KeepTogether, Paragraph, SimpleDocTemplate, Spacer, Table,
                                    TableStyle)
    from xml.sax.saxutils import escape

    PAGINA = "https://servicios.inamhi.gob.ec/pronostico/"
    API = "https://servicios.inamhi.gob.ec/wp-json/inamhi/v1/banner"
    TZ_ECUADOR = ZoneInfo("America/Guayaquil")
    MODULO = "INAMHI Pronóstico"
    PREFIJO = "inamhi_pronostico_"
    PERIODO_REGISTRO = "diario: se reemplaza al día siguiente"
    avisos = []

    sesion = requests.Session()
    sesion.headers.update({
        "User-Agent": "webscraping-anticipa/1.0 (+https://github.com/lreiners-3is/webscraping-anticipa)"
    })

    def pedir(metodo, url, timeout=60, intentos=3):
        ultimo_error = None
        for intento in range(1, intentos + 1):
            try:
                r = sesion.request(metodo, url, timeout=timeout, allow_redirects=True)
                r.raise_for_status()
                return r
            except requests.RequestException as e:
                ultimo_error = e
                print(f"  [aviso] intento {intento}/{intentos} falló para {url}: {e}")
                time.sleep(3 * intento)
        raise RuntimeError(f"No se pudo consultar {url}: {ultimo_error}")

    # ----- Conversión fiel: sin dato o dato no numérico -> None (null) -----
    def numero(valor):
        if valor is None or isinstance(valor, bool):
            return None
        try:
            n = float(str(valor).strip().replace(",", "."))
        except ValueError:
            return None
        if n != n:  # NaN
            return None
        return int(n) if n.is_integer() else n

    def booleano(valor):
        return valor if isinstance(valor, bool) else None

    def texto(valor):
        if valor is None:
            return None
        t = " ".join(str(valor).split())
        return t or None

    def fecha_iso(valor):
        t = texto(valor)
        return t if t and re.fullmatch(r"\d{4}-\d{2}-\d{2}", t) else None

    # ----- 1. Página: secciones (provincias) y boletines regionales -----
    html = pedir("GET", PAGINA).text
    zonas, provincias_pagina = {}, {}
    bloque = re.search(r"const ZONAS\s*=\s*\{(.*?)\};", html, re.S)
    if bloque:
        for clave, titulo, url in re.findall(
                r"(\w+)\s*:\s*\{\s*titulo:\s*'([^']*)'\s*,\s*url:\s*'([^']*)'", bloque.group(1)):
            zonas[clave] = {"titulo": titulo, "url": url}
    nacional = re.search(r"const NACIONAL\s*=\s*\{\s*titulo:\s*'([^']*)'\s*,\s*url:\s*'([^']*)'", html)
    bloque = re.search(r"const PROVINCIAS\s*=\s*\{(.*?)\};", html, re.S)
    if bloque:
        for clave, nombre, zona in re.findall(
                r"'([\w-]+)'\s*:\s*\{\s*nombre:\s*'([^']*)'\s*,\s*zona:\s*(?:'(\w+)'|null)", bloque.group(1)):
            provincias_pagina[nombre] = zona or None
    if not provincias_pagina:
        avisos.append("No se pudo leer la lista de provincias de la página")

    def boletin_de(provincia):
        """Boletín que la página asigna a la provincia (regional, o nacional si no tiene zona)."""
        if provincia not in provincias_pagina:
            return None
        zona = provincias_pagina[provincia]
        if zona and zona in zonas:
            return {"tipo": "regional", "zona": zona, **zonas[zona]}
        if nacional:
            return {"tipo": "nacional", "zona": None, "titulo": nacional.group(1), "url": nacional.group(2)}
        return None

    boletines = []
    for clave, datos in ([(z, zonas[z]) for z in zonas] +
                         ([("nacional", {"titulo": nacional.group(1), "url": nacional.group(2)})] if nacional else [])):
        cubre = sorted((p for p, z in provincias_pagina.items()
                        if (z == clave) or (clave == "nacional" and (z is None or z not in zonas))),
                       key=str.casefold)
        ultima_modificacion = None
        try:
            time.sleep(1)
            lm = pedir("HEAD", datos["url"], timeout=30).headers.get("Last-Modified")
            if lm:
                ultima_modificacion = parsedate_to_datetime(lm).astimezone(TZ_ECUADOR).isoformat(timespec="seconds")
        except Exception as e:  # La fecha del PDF es informativa; su falta no detiene el módulo
            avisos.append(f"Sin fecha para {datos['url']}: {e}")
        boletines.append({"zona": clave, "titulo": datos["titulo"], "url": datos["url"],
                          "provincias": cubre, "ultima_modificacion_servidor": ultima_modificacion})

    # ----- 2. Servicio de datos del mapa -----
    respuesta = pedir("GET", API)
    datos_api = respuesta.json()
    if not isinstance(datos_api, dict):
        raise ValueError("El servicio de pronóstico no devolvió un objeto JSON")
    fecha_publicacion = fecha_iso(datos_api.get("date"))
    if fecha_publicacion is None:
        avisos.append("El servicio no informa la fecha del pronóstico ('date')")

    # Solo el pronóstico de hoy, con la misma regla que el mapa: se usa "today";
    # si una ciudad quedó con la fecha de ayer y "next" ya trae la de hoy (Ecuador), se usa "next".
    hoy_ecuador = datetime.now(TZ_ECUADOR).date().isoformat()
    lista_hoy = [c for c in (datos_api.get("today") or []) if isinstance(c, dict)]
    siguiente_hoy = {texto(c.get("locality_name")): c for c in (datos_api.get("next") or [])
                     if isinstance(c, dict) and c.get("date") == hoy_ecuador}
    lista = [(siguiente_hoy.get(texto(c.get("locality_name"))) if c.get("date") != hoy_ecuador else None) or c
             for c in lista_hoy]
    if not lista:
        lista = list(siguiente_hoy.values())
    if not lista:
        raise ValueError("El servicio de pronóstico no trae localidades para hoy")

    registros = []
    for c in lista:
        periodos = []
        for p in (c.get("forecast") if isinstance(c.get("forecast"), list) else []):
            if not isinstance(p, dict):
                continue
            especial = p.get("special_icon") if isinstance(p.get("special_icon"), dict) else None
            periodos.append({
                "periodo": texto(p.get("period_name")),
                "condicion": texto(p.get("condition_name")),
                "descripcion": texto(p.get("condition_description")),
                "fenomeno_especial": texto(especial.get("name")) if especial else None,
                "inicio": texto(p.get("period_start")),
                "fin": texto(p.get("period_end")),
                "publicado": booleano(p.get("status")),
            })
        registros.append({
            "provincia": texto(c.get("fk_province_name")),
            "provincia_id": c.get("fk_province_id") if isinstance(c.get("fk_province_id"), int) else None,
            "localidad": texto(c.get("locality_name")),
            "localidad_id": c.get("fk_locality_id") if isinstance(c.get("fk_locality_id"), int) else None,
            "fecha_pronostico": fecha_iso(c.get("date")),
            "temperatura_min_c": numero(c.get("min_temperature")),
            "temperatura_max_c": numero(c.get("max_temperature")),
            "indice_uv": numero(c.get("uv_radiation")),
            "lluvia": booleano(c.get("rain")),
            "derivada": booleano(c.get("derivada")),
            "ciudad_base": texto(c.get("ciudad_base")),
            "zona_ciudad": texto(c.get("zona")),
            "nota_fuente": texto(c.get("nota")),
            "latitud": numero(c.get("latitude")),
            "longitud": numero(c.get("longitude")),
            "publicado": booleano(c.get("status")),
            "periodos": periodos,
        })
    if not registros:
        raise ValueError("El servicio de pronóstico no trae registros válidos")

    registros.sort(key=lambda r: ((r["provincia"] or "").casefold(), (r["localidad"] or "").casefold(),
                                  r["fecha_pronostico"] or ""))

    # Provincias: las de la página y las que traiga la fuente. Sin ciudades -> sin valores.
    con_datos = {r["provincia"] for r in registros if r["provincia"]}
    nombres = sorted(set(provincias_pagina) | con_datos, key=str.casefold)
    provincias = []
    for nombre in nombres:
        provincias.append({
            "provincia": nombre,
            "localidades": sorted({r["localidad"] for r in registros if r["provincia"] == nombre and r["localidad"]},
                                  key=str.casefold),
            "tiene_pronostico_por_ciudad": nombre in con_datos,
            "boletin_de_la_pagina": boletin_de(nombre),
        })

    fecha = fecha_publicacion or datetime.now(TZ_ECUADOR).date().isoformat()
    contenido = {
        "registros": registros,
        "provincias": provincias,
        "boletines_regionales": boletines,
    }
    huella = hashlib.sha256(json.dumps(contenido, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()

    # ----- 3. Archivos del día -----
    ruta_json = DIR_INAMHI_PRONOSTICO / f"{PREFIJO}{fecha}.json"
    ruta_pdf = DIR_INAMHI_PRONOSTICO / f"{PREFIJO}{fecha}.pdf"
    existentes = sorted(DIR_INAMHI_PRONOSTICO.glob(f"{PREFIJO}*.*"))
    fechas_existentes = {m.group(1) for p in existentes
                         if (m := re.fullmatch(rf"{PREFIJO}(\d{{4}}-\d{{2}}-\d{{2}})\.(json|pdf)", p.name))}
    if any(f > fecha for f in fechas_existentes):
        avisos.append(f"Ya hay un pronóstico más reciente que {fecha}; no se reemplaza")
        print(f"  [aviso] hay archivos más recientes que {fecha}: no se escribe ni se borra nada")
        return {"fecha_pronostico": fecha, "registros": len(registros), "archivos": "sin cambios",
                "avisos": len(avisos)}

    anterior = None
    if ruta_json.exists() and ruta_pdf.exists():
        try:
            anterior = json.loads(ruta_json.read_text(encoding="utf-8")).get("huella_datos")
        except (ValueError, OSError):
            anterior = None

    if anterior == huella:
        estado_archivos = "sin cambios"
    else:
        ahora = datetime.now(TZ_ECUADOR).isoformat(timespec="seconds")
        documento = {
            "fuente": {
                "institucion": "INAMHI - Instituto Nacional de Meteorología e Hidrología (Ecuador)",
                "pagina": PAGINA,
                "servicio_datos": API,
                "version_servicio": texto(datos_api.get("version")),
            },
            "fecha_publicacion": fecha_publicacion,
            "fecha_publicacion_nota": ("Campo 'date' del servicio de datos. La fuente no publica la hora de "
                                       "emisión. Cada registro trae su propia fecha_pronostico."),
            "fecha_consulta": ahora,
            "unidades": {"temperatura": "°C", "indice_uv": "índice UV"},
            "criterios": [
                "Solo el pronóstico de hoy, el que muestra el mapa de la página.",
                "Solo se copian valores publicados por la fuente; lo no disponible es null.",
                "derivada y ciudad_base son marcas de la fuente: indican que la ciudad usa el pronóstico de otra.",
                "Las provincias sin ciudades en la fuente no tienen valores asignados.",
                "Los boletines regionales se citan, pero no se extraen valores de ellos.",
                "Los horarios de inicio y fin de cada período se copian tal como los publica la fuente.",
            ],
            "huella_datos": huella,
            **contenido,
        }

        # JSON
        temporal = ruta_json.with_name(ruta_json.name + ".part")
        try:
            temporal.write_text(json.dumps(documento, ensure_ascii=False, indent=2), encoding="utf-8")
            os.replace(temporal, ruta_json)
        finally:
            temporal.unlink(missing_ok=True)

        # PDF: por provincia y ciudad; cada ciudad en líneas que se entienden solas.
        estilos = getSampleStyleSheet()
        normal = ParagraphStyle("n", parent=estilos["Normal"], fontSize=9, leading=12)
        pequeño = ParagraphStyle("p", parent=normal, fontSize=8, leading=10, textColor=colors.HexColor("#444444"))
        titulo_prov = ParagraphStyle("tp", parent=estilos["Heading2"], spaceBefore=10, spaceAfter=4)
        titulo_ciudad = ParagraphStyle("tc", parent=estilos["Heading4"], spaceBefore=6, spaceAfter=2)

        def e(valor, sufijo=""):
            if valor is None:
                return "sin dato"
            if isinstance(valor, bool):
                return "sí" if valor else "no"
            if isinstance(valor, float):
                valor = f"{valor:g}".replace(".", ",")
            return escape(f"{valor}{sufijo}")

        historia = [
            Paragraph("Pronóstico del tiempo de hoy por provincia y ciudad - INAMHI Ecuador", estilos["Title"]),
            Paragraph(f"<b>Fecha de publicación (fuente):</b> {e(fecha_publicacion)}", normal),
            Paragraph(f"<b>Fecha de consulta:</b> {e(ahora)}", normal),
            Paragraph(f"<b>Fuente:</b> {escape(PAGINA)} (datos: {escape(API)})", normal),
            Spacer(1, 6),
            Paragraph("Los valores son los publicados por el INAMHI. \"sin dato\" indica que la fuente no "
                      "publica esa variable. Las provincias sin ciudades en la fuente no tienen valores.", pequeño),
        ]
        for prov in provincias:
            nombre = prov["provincia"]
            bloque_prov = [Paragraph(escape(nombre), titulo_prov)]
            b = prov["boletin_de_la_pagina"]
            if b:
                bloque_prov.append(Paragraph(
                    f"Boletín que la página asigna a la provincia: {escape(b['titulo'])} ({escape(b['url'])})", pequeño))
            if not prov["tiene_pronostico_por_ciudad"]:
                bloque_prov.append(Paragraph(
                    f"Provincia: {escape(nombre)} | La fuente no publica pronóstico por ciudad para esta provincia. "
                    "No se asignan valores.", normal))
                historia.append(KeepTogether(bloque_prov))
                continue
            historia.extend(bloque_prov)
            for ciudad in prov["localidades"]:
                filas = [r for r in registros if r["provincia"] == nombre and r["localidad"] == ciudad]
                bloque = [Paragraph(escape(ciudad), titulo_ciudad)]
                base = next((r for r in filas if r["derivada"] or r["ciudad_base"]), None)
                if base:
                    origen = [f"ciudad base: {base['ciudad_base']}" if base["ciudad_base"] else None,
                              f"zona: {base['zona_ciudad']}" if base["zona_ciudad"] else None,
                              "pronóstico derivado de otra ciudad" if base["derivada"] else None,
                              f"nota de la fuente: {base['nota_fuente']}" if base["nota_fuente"] else None]
                    bloque.append(Paragraph("Según la fuente: " + escape("; ".join(o for o in origen if o)) + ".",
                                            pequeño))
                tabla = [["Fecha", "T. mín.", "T. máx.", "UV", "Lluvia"]]
                for r in filas:
                    tabla.append([e(r["fecha_pronostico"]), e(r["temperatura_min_c"], " °C"),
                                  e(r["temperatura_max_c"], " °C"), e(r["indice_uv"]), e(r["lluvia"])])
                t = Table(tabla, colWidths=[2.6 * cm, 2.4 * cm, 2.4 * cm, 1.6 * cm, 1.8 * cm],
                          hAlign="LEFT")
                t.setStyle(TableStyle([
                    ("FONTSIZE", (0, 0), (-1, -1), 8.5),
                    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E8EEF4")),
                    ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#B8C4D0")),
                    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                ]))
                bloque.append(t)
                for r in filas:
                    periodos = "; ".join(
                        f"{p['periodo'] or 'período sin nombre'}: {p['descripcion'] or p['condicion'] or 'sin dato'}"
                        + (f" ({p['fenomeno_especial']})" if p["fenomeno_especial"] else "")
                        for p in r["periodos"]) or "sin dato"
                    bloque.append(Paragraph(
                        f"Provincia: {e(nombre)} | Ciudad: {e(ciudad)} | Fecha: {e(r['fecha_pronostico'])} | "
                        f"Temp. mínima: {e(r['temperatura_min_c'], ' °C')} | "
                        f"Temp. máxima: {e(r['temperatura_max_c'], ' °C')} | UV: {e(r['indice_uv'])} | "
                        f"Lluvia: {e(r['lluvia'])} | Condición por período: {escape(periodos)}", normal))
                historia.append(KeepTogether(bloque))
        if boletines:
            historia.append(Paragraph("Boletines regionales enlazados en la página", titulo_prov))
            historia.append(Paragraph("Se citan como referencia; sus valores no se extraen.", pequeño))
            for b in boletines:
                historia.append(Paragraph(
                    f"{escape(b['titulo'])} | Provincias: {escape(', '.join(b['provincias']) or 'sin dato')} | "
                    f"Última modificación en el servidor: {e(b['ultima_modificacion_servidor'])} | {escape(b['url'])}",
                    normal))

        temporal = ruta_pdf.with_name(ruta_pdf.name + ".part")
        try:
            SimpleDocTemplate(str(temporal), pagesize=A4, leftMargin=1.8 * cm, rightMargin=1.8 * cm,
                              topMargin=1.6 * cm, bottomMargin=1.6 * cm,
                              title=f"Pronóstico INAMHI {fecha}", author="INAMHI (datos)").build(historia)
            os.replace(temporal, ruta_pdf)
        finally:
            temporal.unlink(missing_ok=True)
        estado_archivos = "actualizados" if anterior else "nuevos"

    for ruta in (ruta_json, ruta_pdf):
        registrar(MODULO, "Pronóstico por ciudad", ruta.name, fecha=fecha, periodo=PERIODO_REGISTRO,
                  huella=huella)

    # ----- 4. Borrar los días anteriores -----
    borrados = 0
    for ruta in existentes:
        if ruta.name in (ruta_json.name, ruta_pdf.name):
            continue
        ruta.unlink(missing_ok=True)
        if ruta.suffix in (".json", ".pdf"):
            m = re.search(r"(\d{4}-\d{2}-\d{2})", ruta.name)
            registrar(MODULO, "Pronóstico por ciudad", ruta.name, fecha=m.group(1) if m else "",
                      periodo=PERIODO_REGISTRO, estado="borrado", motivo="Reemplazado por el pronóstico de " + fecha)
            borrados += 1

    for aviso in avisos:
        print("  [aviso]", aviso)
    sin_ciudades = [p["provincia"] for p in provincias if not p["tiene_pronostico_por_ciudad"]]
    print(f"Pronóstico {fecha}: {len(registros)} registros, {len(con_datos)} provincias con ciudades"
          + (f", sin ciudades en la fuente: {', '.join(sin_ciudades)}" if sin_ciudades else "")
          + f" | archivos {estado_archivos} | días anteriores borrados: {borrados}")
    return {"fecha_pronostico": fecha, "registros": len(registros), "provincias_con_ciudades": len(con_datos),
            "archivos": estado_archivos, "borrados": borrados, "avisos": len(avisos)}

## 9. Módulo 7: SENAMHI Alertas

Avisos meteorológicos del SENAMHI de Perú (`www.senamhi.gob.pe/?p=aviso-meteorologico`).

- **Qué avisos:** los que la tabla del SENAMHI marca como `vigente` (en curso) o `emitido` (publicados y por empezar). El historial vencido no se incluye.
- **Qué días:** de cada aviso, solo los días desde hoy en adelante (hora de Perú). Si la fecha de un día no se puede leer, se incluye.
- **Datos:** de la tabla, número, estado, nivel, título y fechas; del detalle, inicio y fin con hora, horas de vigencia y descripción; de cada mapa diario, fecha, horario, nivel del día, texto del día, y departamentos y provincias de posible afectación. Las provincias no tienen nivel propio en la fuente: el nivel es del día.
- **Archivos:** `senamhi_alertas_diaria.json` con todos los avisos activos, y un PDF de solo texto por aviso, `senamhi_aviso_<Nro>.pdf`. Cada archivo se sobrescribe solo si cambia, y el PDF de un aviso se borra cuando deja de estar activo, así que la carpeta nunca acumula el historial.

In [ ]:
# 9. Módulo 7: SENAMHI Alertas
def ejecutar_senamhi():
    import codecs
    import re
    from urllib.parse import parse_qs, urljoin, urlparse
    from xml.sax.saxutils import escape

    import requests
    from bs4 import BeautifulSoup
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import cm
    from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer

    BASE = "https://www.senamhi.gob.pe/"
    LISTA = BASE + "?p=aviso-meteorologico"
    TZ_PERU = ZoneInfo("America/Lima")
    MODULO = "SENAMHI Alertas"
    NOMBRE = "senamhi_alertas_diaria"
    NIVELES = {"2": "AMARILLO", "3": "NARANJA", "4": "ROJO"}  # Códigos del sitio (clases dos/tres/cuatro)
    MESES = {m: i for i, m in enumerate(["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
                                          "septiembre", "octubre", "noviembre", "diciembre"], 1)}
    PAUSA_SEGUNDOS = 1.0
    avisos_internos = []

    sesion = requests.Session()
    sesion.headers.update({
        "User-Agent": "webscraping-anticipa/1.0 (+https://github.com/lreiners-3is/webscraping-anticipa)",
        "Referer": LISTA,
    })

    # El sitio declara UTF-8 pero algunos nombres (Ñ) llegan en Latin-1:
    # se lee como UTF-8 y solo los bytes inválidos se interpretan como Windows-1252.
    def _respaldo_latin(error):
        return error.object[error.start:error.end].decode("cp1252", errors="replace"), error.end

    codecs.register_error("senamhi_latin", _respaldo_latin)

    def pedir(url, timeout=90, intentos=3):
        ultimo_error = None
        for intento in range(1, intentos + 1):
            try:
                time.sleep(PAUSA_SEGUNDOS)
                r = sesion.get(url, timeout=timeout)
                r.raise_for_status()
                return r.content.decode("utf-8", errors="senamhi_latin")
            except requests.RequestException as e:
                ultimo_error = e
                print(f"  [aviso] intento {intento}/{intentos} falló para {url}: {e}")
                time.sleep(3 * intento)
        raise RuntimeError(f"No se pudo consultar {url}: {ultimo_error}")

    def limpio(t):
        t = " ".join((t or "").split())
        return t or None

    def fecha_de_texto(t):
        """'Jueves , 12 de Noviembre de 2026' -> '2026-11-12'; None si no se puede leer."""
        m = re.search(r"(\d{1,2})\s+de\s+([A-Za-zÁÉÍÓÚáéíóú]+)\s+de\s+(\d{4})", t or "")
        if not m or m.group(2).lower() not in MESES:
            return None
        try:
            return datetime(int(m.group(3)), MESES[m.group(2).lower()], int(m.group(1))).date().isoformat()
        except ValueError:
            return None

    def fecha_iso(t):
        t = limpio(t)
        return t if t and re.fullmatch(r"\d{4}-\d{2}-\d{2}", t) else None

    hoy = datetime.now(TZ_PERU).date().isoformat()

    # ----- 1. Tabla general: avisos vigentes y emitidos -----
    tabla = BeautifulSoup(pedir(LISTA), "html.parser").find("table")
    if tabla is None:
        raise RuntimeError("No se encontró la tabla de avisos en la página")
    filas_tabla = 0
    activos = []
    for fila in tabla.find_all("tr"):
        celdas = fila.find_all("td")
        if len(celdas) < 7:
            continue
        filas_tabla += 1
        estado = (celdas[0].get("class") or [None])[0]
        if estado not in ("vigente", "emitido"):
            continue
        numero = re.match(r"\d+", celdas[1].get_text(strip=True))
        enlace = celdas[0].find("a")
        duracion = re.search(r"\d+", celdas[5].get_text())
        activos.append({
            "numero": int(numero.group()) if numero else None,
            "estado": estado,
            "nivel": limpio(celdas[6].get_text()),
            "titulo": limpio(celdas[0].get_text(" ")),
            "fecha_emision": fecha_iso(celdas[2].get_text()),
            "fecha_inicio": fecha_iso(celdas[3].get_text()),
            "fecha_fin": fecha_iso(celdas[4].get_text()),
            "duracion_horas": int(duracion.group()) if duracion else None,
            "inicio_evento": None,
            "fin_evento": None,
            "descripcion": None,
            "url": urljoin(BASE, enlace["href"]) if enlace and enlace.get("href") else None,
            "dias": [],
        })
    if not filas_tabla:
        raise RuntimeError("La tabla de avisos está vacía")

    # ----- 2. Detalle: una página muestra todos los avisos activos -----
    detalle = None
    url_detalle = next((a["url"] for a in activos if a["url"]), None)
    if url_detalle:
        detalle = BeautifulSoup(pedir(url_detalle), "html.parser")

    for aviso in activos:
        bloque = None
        if detalle is not None and aviso["numero"] is not None:
            h2 = next((h for h in detalle.find_all("h2")
                       if re.search(rf"N°\s*{aviso['numero']}\b", h.get_text(" "))), None)
            bloque = h2.parent if h2 else None
        if bloque is None:
            avisos_internos.append(f"Sin detalle para el aviso {aviso['numero']}")
            continue
        texto_bloque = limpio(bloque.get_text(" ")) or ""
        m = re.search(r"Inicio del evento:\s*(.*?horas)", texto_bloque)
        aviso["inicio_evento"] = limpio(m.group(1)) if m else None
        m = re.search(r"Fin del evento:\s*(.*?horas)", texto_bloque)
        aviso["fin_evento"] = limpio(m.group(1)) if m else None
        for div in bloque.find_all("div", class_="col-lg-12"):
            t = limpio(div.get_text(" "))
            if t and t.startswith("El SENAMHI"):
                aviso["descripcion"] = t
                break

        # ----- 3. Un mapa por día -----
        for pane in bloque.select("div.tab-pane"):
            marco = pane.find("iframe")
            if not marco or not marco.get("src"):
                continue
            url_mapa = urljoin(BASE, marco["src"])
            parametros = {k: v[0] for k, v in parse_qs(urlparse(url_mapa).query).items()}
            parrafo = pane.find("p")
            dia = {
                "numero_dia": int(parametros["mp"]) if parametros.get("mp", "").isdigit() else None,
                "fecha": None,
                "fecha_texto": None,
                "hora_inicio": None,
                "hora_fin": None,
                "nivel": NIVELES.get(parametros.get("nl")),
                "texto": limpio(parrafo.get_text(" ")) if parrafo else None,
                "departamentos": [],
                "provincias_por_departamento": {},
                "mapa_url": url_mapa,
            }
            try:
                mapa = BeautifulSoup(pedir(url_mapa), "html.parser")
                for fila_mapa in mapa.select(".row"):
                    if limpio(fila_mapa.get_text(" ")) == "Fecha:":
                        siguiente = fila_mapa.find_next_sibling("div")
                        dia["fecha_texto"] = limpio(siguiente.get_text(" ")) if siguiente else None
                        break
                dia["fecha"] = fecha_de_texto(dia["fecha_texto"])
                texto_mapa = limpio(mapa.get_text(" ")) or ""
                m = re.search(r"Hora de inicio:\s*([\d:]+)", texto_mapa)
                dia["hora_inicio"] = m.group(1) if m else None
                m = re.search(r"Hora de fin:\s*([\d:]+)", texto_mapa)
                dia["hora_fin"] = m.group(1) if m else None
                nombres = {}
                for d in mapa.select("a.depBuscarZoom"):
                    nombre = limpio(d.get_text())
                    if nombre:
                        dia["departamentos"].append(nombre)
                        nombres[d.get("id")] = nombre
                for p in mapa.select("a.depTotales"):
                    clave = next((c[4:] for c in p.get("class", []) if c.startswith("dep_")), None)
                    provincia = (limpio(p.get_text()) or "").rstrip(",").strip()
                    if provincia:
                        dia["provincias_por_departamento"].setdefault(nombres.get(clave, clave), []).append(provincia)
            except Exception as e:  # Un mapa caído no elimina el día: queda con lo que se pudo leer
                avisos_internos.append(f"Mapa no disponible ({url_mapa}): {e}")
            # Solo días desde hoy; si la fecha no se pudo leer, se incluye.
            if dia["fecha"] is None or dia["fecha"] >= hoy:
                aviso["dias"].append(dia)

    # Los vigentes que ya no tienen días desde hoy (terminaron) no se incluyen.
    activos = [a for a in activos if a["dias"] or a["estado"] == "emitido" or (a["fecha_fin"] or hoy) >= hoy]
    activos.sort(key=lambda a: (a["fecha_inicio"] or "", a["numero"] or 0))

    # ----- 4. Archivos: un JSON con todos los avisos y un PDF por aviso -----
    def huella_de(dato):
        return hashlib.sha256(json.dumps(dato, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()

    for a in activos:
        a["archivo_pdf"] = f"senamhi_aviso_{a['numero']}.pdf" if a["numero"] is not None else None
        a["huella"] = huella_de(a)
    huella = huella_de({"avisos": activos})
    ruta_json = DIR_SENAMHI / f"{NOMBRE}.json"

    # Huellas de la corrida anterior: del JSON completo y de cada PDF.
    huella_json_previa, huellas_pdf_previas = None, {}
    if ruta_json.exists():
        try:
            previo = json.loads(ruta_json.read_text(encoding="utf-8"))
            huella_json_previa = previo.get("huella_datos")
            huellas_pdf_previas = {a.get("archivo_pdf"): a.get("huella") for a in previo.get("avisos", [])}
        except (ValueError, OSError):
            pass

    ahora = datetime.now(TZ_PERU).isoformat(timespec="seconds")
    if huella_json_previa == huella:
        estado_json = "sin cambios"
    else:
        documento = {
            "fuente": {"institucion": "SENAMHI - Servicio Nacional de Meteorología e Hidrología del Perú",
                       "pagina": LISTA},
            "fecha_consulta": ahora,
            "fecha_referencia_peru": hoy,
            "criterios": [
                "Avisos con estado 'vigente' o 'emitido' en la tabla del SENAMHI.",
                "De cada aviso, solo los días desde la fecha de referencia (hora de Perú).",
                "Valores copiados de la fuente; lo no disponible es null.",
                "Las provincias no tienen nivel propio en la fuente: el nivel es del día.",
                "Cada aviso tiene su PDF (archivo_pdf): se sobrescribe si el aviso cambia y se borra al dejar de estar activo.",
            ],
            "huella_datos": huella,
            "total_avisos": len(activos),
            "avisos": activos,
        }
        temporal = ruta_json.with_name(ruta_json.name + ".part")
        try:
            temporal.write_text(json.dumps(documento, ensure_ascii=False, indent=2), encoding="utf-8")
            os.replace(temporal, ruta_json)
        finally:
            temporal.unlink(missing_ok=True)
        estado_json = "actualizado" if huella_json_previa else "nuevo"
    registrar(MODULO, "Avisos activos", ruta_json.name, fecha=hoy, periodo="se sobrescribe en cada corrida",
              huella=huella)

    # PDF de un aviso: solo texto, en líneas que se entienden solas.
    estilos = getSampleStyleSheet()
    normal = ParagraphStyle("n", parent=estilos["Normal"], fontSize=9, leading=12, spaceAfter=3)
    nota = ParagraphStyle("p", parent=normal, fontSize=8, leading=10, textColor=colors.HexColor("#444444"))
    t_aviso = ParagraphStyle("ta", parent=estilos["Title"], fontSize=15, leading=19)
    t_seccion = ParagraphStyle("ts", parent=estilos["Heading3"], spaceBefore=8, spaceAfter=3)
    t_dia = ParagraphStyle("td", parent=estilos["Heading4"], spaceBefore=6, spaceAfter=2)

    def e(valor):
        return "sin dato" if valor is None else escape(str(valor))

    def pdf_de_aviso(a, ruta):
        cab = f"Aviso N° {e(a['numero'])}"
        historia = [
            Paragraph(f"{cab} - {e(a['nivel'])}<br/>{e(a['titulo'])}", t_aviso),
            Paragraph(f"<b>Fuente:</b> SENAMHI (Perú) - {escape(a['url'] or LISTA)}", nota),
            Paragraph(f"<b>Fecha de consulta:</b> {e(ahora)}. Se incluyen solo los días desde {e(hoy)} "
                      "(hora de Perú). Las provincias no tienen nivel propio en la fuente: el nivel es del día.", nota),
            Spacer(1, 4),
            Paragraph("Datos del aviso", t_seccion),
            Paragraph(f"{cab} | Estado: {e(a['estado'])} | Nivel: {e(a['nivel'])} | Título: {e(a['titulo'])}",
                      normal),
            Paragraph(f"{cab} | Fecha de emisión: {e(a['fecha_emision'])} | "
                      f"Inicio del evento: {e(a['inicio_evento'] or a['fecha_inicio'])} | "
                      f"Fin del evento: {e(a['fin_evento'] or a['fecha_fin'])} | "
                      f"Periodo de vigencia: {e(a['duracion_horas'])} horas", normal),
        ]
        if a["descripcion"]:
            historia += [Paragraph("Descripción", t_seccion), Paragraph(f"{cab} | {e(a['descripcion'])}", normal)]
        if a["dias"]:
            historia.append(Paragraph("Pronóstico por día", t_seccion))
        else:
            historia.append(Paragraph(f"{cab} | El aviso no tiene días desde {e(hoy)} en la fuente.", normal))
        for d in a["dias"]:
            historia.append(Paragraph(f"{e(d['fecha_texto'] or d['fecha'])} - nivel {e(d['nivel'])}", t_dia))
            historia.append(Paragraph(
                f"{cab} | Día: {e(d['fecha'])} ({e(d['hora_inicio'])} a {e(d['hora_fin'])}, hora local) | "
                f"Nivel del día: {e(d['nivel'])} | {e(d['texto'])}", normal))
            historia.append(Paragraph(
                f"{cab} | Día: {e(d['fecha'])} | Departamentos de posible afectación: "
                f"{e(', '.join(d['departamentos']) or None)}", normal))
            for dep, provs in d["provincias_por_departamento"].items():
                historia.append(Paragraph(
                    f"{cab} | Día: {e(d['fecha'])} | Nivel: {e(d['nivel'])} | Departamento: {e(dep)} | "
                    f"Provincias de posible afectación: {e(', '.join(provs))}", normal))
        temporal = ruta.with_name(ruta.name + ".part")
        try:
            SimpleDocTemplate(str(temporal), pagesize=A4, leftMargin=1.8 * cm, rightMargin=1.8 * cm,
                              topMargin=1.6 * cm, bottomMargin=1.6 * cm,
                              title=f"SENAMHI {cab}", author="SENAMHI (datos)").build(historia)
            os.replace(temporal, ruta)
        finally:
            temporal.unlink(missing_ok=True)

    pdf_nuevos = pdf_actualizados = pdf_sin_cambios = 0
    pdf_activos = set()
    for a in activos:
        if not a["archivo_pdf"]:
            avisos_internos.append(f"Aviso sin número: no se genera su PDF ({a['titulo']})")
            continue
        ruta = DIR_SENAMHI / a["archivo_pdf"]
        pdf_activos.add(ruta.name)
        if ruta.exists() and huellas_pdf_previas.get(ruta.name) == a["huella"]:
            pdf_sin_cambios += 1
        else:
            existia = ruta.exists()
            pdf_de_aviso(a, ruta)
            if existia:
                pdf_actualizados += 1
            else:
                pdf_nuevos += 1
        registrar(MODULO, f"Aviso {a['numero']}", ruta.name, fecha=a["fecha_emision"] or hoy,
                  periodo="se sobrescribe si el aviso cambia; se borra al dejar de estar activo",
                  motivo=f"{a['estado']} | {a['nivel']} | {a['titulo']}", huella=a["huella"])

    # Se borran los PDF de avisos que ya no están activos (y un PDF consolidado, si quedara).
    pdf_borrados = 0
    for ruta in list(DIR_SENAMHI.glob("senamhi_aviso_*.pdf")) + [DIR_SENAMHI / f"{NOMBRE}.pdf"]:
        if ruta.exists() and ruta.name not in pdf_activos:
            ruta.unlink()
            pdf_borrados += 1
            registrar(MODULO, "Avisos activos", ruta.name, fecha=hoy, estado="borrado",
                      periodo="se borra al dejar de estar activo", motivo="El aviso ya no está vigente ni emitido")

    for aviso in avisos_internos:
        print("  [aviso]", aviso)
    dias = sum(len(a["dias"]) for a in activos)
    print(f"SENAMHI: {len(activos)} avisos activos ({', '.join(str(a['numero']) for a in activos) or 'ninguno'}), "
          f"{dias} días desde {hoy} | JSON {estado_json} | PDF nuevos {pdf_nuevos}, actualizados {pdf_actualizados}, "
          f"sin cambios {pdf_sin_cambios}, borrados {pdf_borrados}")
    return {"avisos_activos": len(activos), "dias": dias, "json": estado_json, "pdf_nuevos": pdf_nuevos,
            "pdf_actualizados": pdf_actualizados, "pdf_sin_cambios": pdf_sin_cambios, "pdf_borrados": pdf_borrados,
            "avisos": len(avisos_internos)}

## 10. Ejecutar

In [ ]:
# 10. Ejecutar los módulos y guardar el resumen
import pandas as pd

MODULOS = [
    ("1. OSPA IDEAM", EJECUTAR_OSPA, ejecutar_ospa, DIR_OSPA),
    ("2. Boletines IDEAM", EJECUTAR_BOLETINES, ejecutar_boletines, DIR_BOLETINES),
    ("3. Monitor Amazonía", EJECUTAR_MONITOR, ejecutar_monitor, DIR_MONITOR),
    ("4. INAMHI El Niño", EJECUTAR_INAMHI, ejecutar_inamhi_elnino, DIR_INAMHI),
    ("5. INAMHI Boletines", EJECUTAR_INAMHI_BOLETINES, ejecutar_inamhi_boletines, DIR_INAMHI_BOLETINES),
    ("6. INAMHI Pronóstico", EJECUTAR_INAMHI_PRONOSTICO, ejecutar_inamhi_pronostico, DIR_INAMHI_PRONOSTICO),
    ("7. SENAMHI Alertas", EJECUTAR_SENAMHI, ejecutar_senamhi, DIR_SENAMHI),
]

resumen_modulos = []

for nombre, activo, funcion, carpeta in MODULOS:
    fila = {"Módulo": nombre, "Estado": "OMITIDO", "Segundos": 0,
            "Detalle": "", "Carpeta": str(carpeta)}

    if activo:
        print(f"\n{'=' * 60}\n{nombre}\n{'=' * 60}")
        inicio = time.time()
        try:
            resultado = funcion()
            fila["Estado"] = "CON AVISOS" if resultado.get("avisos") else "OK"
            fila["Detalle"] = ", ".join(f"{k}: {v}" for k, v in resultado.items())
        except Exception as e:  # Un módulo caído no detiene a los demás
            traceback.print_exc()
            fila["Estado"] = "FALLÓ"
            fila["Detalle"] = f"{type(e).__name__}: {e}"[:500]
        fila["Segundos"] = round(time.time() - inicio)

    resumen_modulos.append(fila)

guardar_registro()

with open(OUTPUT / "resumen_ejecucion.json", "w", encoding="utf-8") as f:
    json.dump({
        "fecha_colombia": datetime.now(ZoneInfo("America/Bogota")).isoformat(),
        "modulos": resumen_modulos,
    }, f, ensure_ascii=False, indent=2)

print(f"\n{'=' * 60}\nRESUMEN\n{'=' * 60}")
print("Salidas en:", OUTPUT)
print("Registro de fechas:", REGISTRO_PATH)

por_estado = {}
for f in REGISTRO.values():
    por_estado[f["estado"]] = por_estado.get(f["estado"], 0) + 1
print("Archivos por estado:", por_estado)

if not BORRAR_ARCHIVOS and por_estado.get("por_borrar"):
    print(f"SIMULACIÓN: se borrarían {por_estado['por_borrar']} archivos. "
          "Revisa las filas 'por_borrar' del registro y activa BORRAR_ARCHIVOS = True.")

display(pd.DataFrame(resumen_modulos))